# 기존 Drive·GitHub 자료 확인
- 모델, 후보 JSON, 기존 평가 데이터, 코드 경로와 구조를 점검

In [1]:

from google.colab import drive
from pathlib import Path
import torch

print("===== 1단계. Google Drive 연결 =====")

drive.mount("/content/drive")

print("\n===== 2단계. 실행 환경 확인 =====")

print("PyTorch:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n===== 3단계. 기존 PinPoint 자료 확인 =====")

paths = {
    "PinPoint Google Drive": Path(
        "/content/drive/MyDrive/pinpoint_capstone"
    ),
    "GitHub 작업 폴더": Path(
        "/content/pinpoint-capstone"
    ),
}

for name, path in paths.items():
    print(f"\n[{name}]")
    print("경로:", path)
    print("존재 여부:", path.exists())

    if path.exists():
        items = sorted(path.iterdir(), key=lambda p: p.name)
        for item in items[:30]:
            kind = "폴더" if item.is_dir() else "파일"
            print(f"  [{kind}] {item.name}")

        if len(items) > 30:
            print(f"  ... 외 {len(items) - 30}개")

print("\n===== 1단계 확인 완료 =====")


===== 1단계. Google Drive 연결 =====
Mounted at /content/drive

===== 2단계. 실행 환경 확인 =====
PyTorch: 2.11.0+cu130
GPU 사용 가능: True
GPU: Tesla T4

===== 3단계. 기존 PinPoint 자료 확인 =====

[PinPoint Google Drive]
경로: /content/drive/MyDrive/pinpoint_capstone
존재 여부: True
  [폴더] 01_AI_인수인계
  [폴더] 02_Korean_Video_Test
  [폴더] 03_Korean_Training_Data
  [폴더] 04_
  [파일] PinPoint_ConsensusV3_MultilingualCLIP.ipynb
  [폴더] data
  [폴더] notebooks

[GitHub 작업 폴더]
경로: /content/pinpoint-capstone
존재 여부: False

===== 1단계 확인 완료 =====


In [2]:

from pathlib import Path

print("===== 2단계. 새 실험 저장 폴더 생성 =====")

BASE_DIR = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "04_PinPoint_ConsensusV3_MultilingualCLIP"
)

folders = [
    "01_code",
    "02_results",
    "03_evaluation",
    "04_models",
    "05_logs",
]

BASE_DIR.mkdir(parents=True, exist_ok=True)

for folder in folders:
    path = BASE_DIR / folder
    path.mkdir(parents=True, exist_ok=True)
    print(f"✅ {path}")

print("\n===== 기존 실험 자료 확인 =====")

old_dir = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test"
)

if old_dir.exists():
    for item in sorted(old_dir.iterdir(), key=lambda x: x.name):
        print(
            "[폴더]" if item.is_dir() else "[파일]",
            item.name
        )
else:
    print("⚠️ 기존 실험 폴더를 찾지 못했습니다.")

print("\n===== 2단계 완료 =====")

===== 2단계. 새 실험 저장 폴더 생성 =====
✅ /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/01_code
✅ /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/02_results
✅ /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/03_evaluation
✅ /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/04_models
✅ /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/05_logs

===== 기존 실험 자료 확인 =====
[폴더] AI_Backup
[파일] PinPoint_Korean_Video_Test.ipynb
[폴더] __pycache__
[폴더] clip_features
[파일] detergent_evaluation.json
[파일] fusion_baseline_evaluation.json
[파일] fusion_experiment.py
[파일] indexer_before_subtitle_integration.py
[파일] kr_daiso_001_subtitles.json
[파일] kr_kitchen_001_subtitles.json
[파일] kr_tofu_001_subtitles.json
[파일] pinpoint_ai_experiment_summary.json
[파일] rice_washing_evaluation.json
[파일] run_moment_detr_inference_before_fusion.py
[파일] search_api_before_

In [4]:

from pathlib import Path
import json

print("===== 3단계. 기존 실험 JSON 탐색 =====")

ROOT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test"
)

KEYWORDS = [
    "consensus",
    "fusion",
    "minilm",
    "comparison",
    "evaluation",
    "summary",
]

results = []

for path in ROOT.rglob("*.json"):
    if any(word in path.name.lower() for word in KEYWORDS):
        results.append(path)

results = sorted(results, key=lambda p: str(p))

print("발견된 관련 JSON:", len(results), "개")

for idx, path in enumerate(results, 1):
    print(f"\n[{idx}] {path.relative_to(ROOT)}")

    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict):
            print("형식: dict")
            print("최상위 키:", list(data.keys())[:25])

            for key, value in list(data.items())[:8]:
                if isinstance(value, list):
                    print(f"  {key}: list ({len(value)}개)")
                    if value and isinstance(value[0], dict):
                        print(
                            "    첫 항목 키:",
                            list(value[0].keys())[:20]
                        )

                elif isinstance(value, dict):
                    print(
                        f"  {key}: dict, 키:",
                        list(value.keys())[:12]
                    )

                else:
                    print(f"  {key}: {str(value)[:100]}")

        elif isinstance(data, list):
            print("형식: list")
            print("항목 수:", len(data))

            if data and isinstance(data[0], dict):
                print("첫 항목 키:", list(data[0].keys())[:25])

        else:
            print("형식:", type(data).__name__)

    except Exception as e:
        print("읽기 오류:", type(e).__name__, str(e)[:150])

print("\n===== 3단계 확인 완료 =====")

===== 3단계. 기존 실험 JSON 탐색 =====
발견된 관련 JSON: 21 개

[1] AI_Backup/minilm_full_validation_metrics_restored_20261010.json
형식: dict
최상위 키: ['task', 'dataset', 'sample_size', 'evaluation_note', 'base_model', 'original', 'finetuned', 'verification']
  task: Korean summary-to-transcript text retrieval
  dataset: AI-Hub validation pairs
  sample_size: 1948
  evaluation_note: Each query is matched against validation transcripts. The transcript at the same index is treated as
  base_model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  original: dict, 키: ['recall_at_1', 'recall_at_5', 'mrr']
  finetuned: dict, 키: ['recall_at_1', 'recall_at_5', 'mrr']
  verification: Metrics reproduced on the full 1,948-pair validation dataset in the earlier Colab experiment.

[2] AI_Backup/public_experiment_results_20261010/consensus_6queries_comparison.json
형식: dict
최상위 키: ['method', 'summary', 'results']
  method: [REDACTED]
  summary: dict, 키: ['mean_top1_iou', 'mean_best5_iou', 'r1_at_05', 'r5_

In [5]:

from pathlib import Path
import json

print("===== 4단계. Consensus V3 원본 후보 구조 확인 =====")

ROOT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "public_experiment_results_20261010"
)

TARGET_FILES = [
    "consensus_6queries_comparison.json",
    "consensus_v2_12queries_comparison.json",
    "minilm_fusion_6queries_comparison.json",
    "squat_4methods_comparison.json",
]

def inspect_structure(obj, depth=0, max_depth=5):
    """값 전체를 출력하지 않고 JSON 구조와 예시를 확인."""
    indent = "  " * depth

    if depth > max_depth:
        print(indent + "... 깊이 제한")
        return

    if isinstance(obj, dict):
        print(indent + f"dict ({len(obj)} keys)")

        for key, value in obj.items():
            print(indent + f"  [{key}]")
            inspect_structure(value, depth + 1, max_depth)

    elif isinstance(obj, list):
        print(indent + f"list ({len(obj)} items)")

        if obj:
            print(indent + "  첫 번째 항목:")
            inspect_structure(obj[0], depth + 1, max_depth)

    else:
        value = str(obj)

        if len(value) > 160:
            value = value[:160] + "..."

        print(indent + f"{type(obj).__name__}: {value}")


for filename in TARGET_FILES:
    path = ROOT / filename

    print("\n" + "=" * 65)
    print("파일:", filename)
    print("존재:", path.exists())

    if not path.exists():
        continue

    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        inspect_structure(data)

    except Exception as e:
        print("읽기 오류:", type(e).__name__, str(e))

print("\n===== 4단계 확인 완료 =====")

===== 4단계. Consensus V3 원본 후보 구조 확인 =====

파일: consensus_6queries_comparison.json
존재: True
dict (3 keys)
  [method]
  str: [REDACTED]
  [summary]
  dict (5 keys)
    [mean_top1_iou]
    float: 0.29695030467407163
    [mean_best5_iou]
    float: 0.6742722657970684
    [r1_at_05]
    int: 2
    [r5_at_05]
    int: 5
    [total_queries]
    int: 6
  [results]
  list (6 items)
    첫 번째 항목:
    dict (5 keys)
      [name]
      str: case_004
      [video_id]
      str: video_003
      [top1_iou]
      float: 0.0
      [best5_iou]
      float: 0.3127272727272727
      [predictions]
      list (5 items)
        첫 번째 항목:
        dict (11 keys)
          [video_id]
          str: video_003
          [query]
          str: query_013
          [start_time]
          float: 98.5
          [end_time]
          float: 118.9581
          [score]
          float: 0.9842
          [source]
          str: visual
          [subtitle_score]
          NoneType: None
          [rank_source]
          str: hy

In [6]:

from pathlib import Path

print("===== 5단계. 기존 실험 코드 및 원본 후보 파일 탐색 =====")

ROOT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test"
)

SEARCH_KEYWORDS = [
    "fusion",
    "consensus",
    "candidate",
    "minilm",
    "search",
    "inference",
    "moment",
]

EXTENSIONS = {
    ".py", ".json", ".jsonl", ".npz", ".pkl",
    ".pickle", ".pt", ".pth"
}

matches = []

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in EXTENSIONS:
        continue

    filename = path.name.lower()

    if any(keyword in filename for keyword in SEARCH_KEYWORDS):
        matches.append(path)

matches.sort(key=lambda p: str(p))

print("발견된 관련 파일:", len(matches), "개")

for i, path in enumerate(matches, 1):
    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"[{i}] {path.relative_to(ROOT)} "
        f"({size_mb:.2f} MB)"
    )

print("\n===== 주요 Python 파일 내 함수 확인 =====")

python_files = [
    ROOT / "fusion_experiment.py",
    ROOT / "run_moment_detr_inference_before_fusion.py",
    ROOT / "search_api_before_fusion.py",
]

for path in python_files:
    print("\n" + "=" * 60)
    print("파일:", path.name)

    if not path.exists():
        print("파일 없음")
        continue

    source = path.read_text(encoding="utf-8")

    for line_number, line in enumerate(source.splitlines(), 1):
        stripped = line.strip()

        if (
            stripped.startswith("def ")
            or stripped.startswith("async def ")
            or stripped.startswith("class ")
        ):
            print(f"{line_number}: {stripped[:150]}")

print("\n===== 5단계 확인 완료 =====")

===== 5단계. 기존 실험 코드 및 원본 후보 파일 탐색 =====
발견된 관련 파일: 23 개
[1] AI_Backup/consensus_rerank.py (0.00 MB)
[2] AI_Backup/fusion_search_20261010.py (0.01 MB)
[3] AI_Backup/fusion_search_FINAL_restored_20261010.py (0.01 MB)
[4] AI_Backup/fusion_search_before_model_connection_20261010_130236.py (0.01 MB)
[5] AI_Backup/fusion_search_before_model_selection_20261010_130203.py (0.01 MB)
[6] AI_Backup/minilm_evaluate_restored_20261010.py (0.00 MB)
[7] AI_Backup/minilm_full_validation_metrics_restored_20261010.json (0.00 MB)
[8] AI_Backup/minilm_prepare_data_restored_20261010.py (0.01 MB)
[9] AI_Backup/minilm_train_restored_20261010.py (0.00 MB)
[10] AI_Backup/public_experiment_results_20261010/consensus_6queries_comparison.json (0.01 MB)
[11] AI_Backup/public_experiment_results_20261010/consensus_v2_12queries_comparison.json (0.02 MB)
[12] AI_Backup/public_experiment_results_20261010/fusion_baseline_evaluation.json (0.00 MB)
[13] AI_Backup/public_experiment_results_20261010/fusion_new_3queries_compar

In [7]:

from pathlib import Path
import ast

print("===== 6단계. Consensus 및 Fusion 코드 구조 확인 =====")

ROOT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup"
)

TARGET_FILES = [
    "consensus_rerank.py",
    "fusion_search_FINAL_restored_20261010.py",
    "fusion_search_20261010.py",
]

for filename in TARGET_FILES:
    path = ROOT / filename

    print("\n" + "=" * 70)
    print("파일:", filename)
    print("존재 여부:", path.exists())

    if not path.exists():
        continue

    source = path.read_text(encoding="utf-8")

    print("코드 전체 줄 수:", len(source.splitlines()))

    try:
        tree = ast.parse(source)

        print("\n[Import 목록]")

        for node in tree.body:
            if isinstance(node, ast.Import):
                for alias in node.names:
                    print("import", alias.name)

            elif isinstance(node, ast.ImportFrom):
                print(
                    "from",
                    node.module,
                    "import",
                    ", ".join(a.name for a in node.names)
                )

        print("\n[함수 목록 및 인자]")

        for node in tree.body:
            if isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef)
            ):
                args = [arg.arg for arg in node.args.args]

                print(
                    f"{node.lineno}: "
                    f"{node.name}({', '.join(args)})"
                )

        print("\n[상위 함수 호출 관련 코드]")

        keywords = [
            "predict_moments",
            "get_multilingual",
            "get_semantic",
            "rank_by_consensus",
            "rank_source",
            "top_k",
        ]

        for line_num, line in enumerate(
            source.splitlines(), 1
        ):
            stripped = line.strip()

            if any(word in stripped for word in keywords):
                print(f"{line_num}: {stripped[:180]}")

    except SyntaxError as e:
        print("문법 분석 오류:", e)

print("\n===== 6단계 확인 완료 =====")

===== 6단계. Consensus 및 Fusion 코드 구조 확인 =====

파일: consensus_rerank.py
존재 여부: True
코드 전체 줄 수: 75

[Import 목록]
from ai.features.fusion_search import temporal_iou

[함수 목록 및 인자]
14: _rank_by_consensus(hybrid, ko, en, min_overlap)
62: rank_by_consensus(hybrid, ko, en)
70: rank_by_consensus_v2(hybrid, ko, en)

[상위 함수 호출 관련 코드]
14: def _rank_by_consensus(hybrid, ko, en, min_overlap=0.0):
27: item["rank_source"] = source
37: if source == candidate["rank_source"]:
62: def rank_by_consensus(hybrid, ko, en):
64: return _rank_by_consensus(
70: def rank_by_consensus_v2(hybrid, ko, en):
72: return _rank_by_consensus(

파일: fusion_search_FINAL_restored_20261010.py
존재 여부: True
코드 전체 줄 수: 270

[Import 목록]
from pathlib import Path
from functools import lru_cache
from sentence_transformers import SentenceTransformer
from sentence_transformers import util
from ai.features.run_moment_detr_inference import predict_moments_hybrid

[함수 목록 및 인자]
7: get_semantic_model(model_type)
44: temporal_iou(a, b)
59: get_s

In [8]:

from pathlib import Path

print("===== 7단계. 기존 후보 생성 및 Consensus 코드 확인 =====")

ROOT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup"
)

FILES_AND_RANGES = {
    "consensus_rerank.py": [
        (1, 75),
    ],
    "fusion_search_FINAL_restored_20261010.py": [
        (1, 43),
        (59, 106),
        (107, 163),
        (213, 270),
    ],
}

for filename, ranges in FILES_AND_RANGES.items():
    path = ROOT / filename

    print("\n" + "=" * 75)
    print("파일:", filename)

    if not path.exists():
        print("파일을 찾을 수 없습니다.")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    for start, end in ranges:
        print(f"\n--- {start}~{end}행 ---")

        for line_number in range(start, min(end, len(lines)) + 1):
            print(
                f"{line_number:3d}: {lines[line_number - 1]}"
            )

print("\n===== 7단계 완료 =====")

===== 7단계. 기존 후보 생성 및 Consensus 코드 확인 =====

파일: consensus_rerank.py

--- 1~75행 ---
  1: """
  2: PinPoint Consensus Reranking
  3: 
  4: V1: 서로 다른 검색 방식의 시간 구간 겹침을 모두 가점
  5: V2: IoU 0.5 이상인 겹침만 가점
  6: 
  7: 실험용 재정렬 알고리즘이며,
  8: 정답 구간(Ground Truth)은 순위 결정에 사용하지 않는다.
  9: """
 10: 
 11: from ai.features.fusion_search import temporal_iou
 12: 
 13: 
 14: def _rank_by_consensus(hybrid, ko, en, min_overlap=0.0):
 15:     groups = {
 16:         "hybrid": hybrid,
 17:         "ko": ko,
 18:         "en": en
 19:     }
 20: 
 21:     candidates = []
 22: 
 23:     for source, group in groups.items():
 24:         for rank, candidate in enumerate(group, 1):
 25:             item = dict(candidate)
 26: 
 27:             item["rank_source"] = source
 28:             item["original_rank"] = rank
 29:             item["rank_score"] = 1.0 / rank
 30: 
 31:             candidates.append(item)
 32: 
 33:     for candidate in candidates:
 34:         support = 0.0
 35: 
 36:         for source, gro

# Consensus V3 구현
- 세 가지 검색 방식의 Top-3와 모델 간 중첩을 이용한 재순위화

In [9]:

from pathlib import Path
import ast

print("===== 8단계. Moment-DETR 원본 후보 추출 함수 확인 =====")

ROOT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test"
)

TARGET = ROOT / "run_moment_detr_inference_before_fusion.py"

if not TARGET.exists():
    raise FileNotFoundError(f"파일이 없습니다: {TARGET}")

source = TARGET.read_text(encoding="utf-8")
lines = source.splitlines()
tree = ast.parse(source)

TARGET_FUNCTIONS = [
    "predict_moments",
    "predict_moment_hybrid",
    "predict_moments_hybrid",
]

for node in tree.body:
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        continue

    if node.name not in TARGET_FUNCTIONS:
        continue

    print("\n" + "=" * 70)
    print("함수:", node.name)
    print("시작 행:", node.lineno)
    print("종료 행:", node.end_lineno)

    args = [arg.arg for arg in node.args.args]
    print("인자:", args)

    print("\n[함수 시작 부분]")

    for n in range(node.lineno, min(node.lineno + 35, node.end_lineno + 1)):
        print(f"{n:4d}: {lines[n-1]}")

    print("\n[후보 생성 및 반환 관련 코드]")

    keywords = [
        '"source"',
        "'source'",
        '"score"',
        "'score'",
        '"start_time"',
        "'start_time'",
        '"end_time"',
        "'end_time'",
        "return ",
        "append(",
        "top_k",
        "_temporal_nms",
    ]

    matched = []

    for n in range(node.lineno, node.end_lineno + 1):
        line = lines[n-1]

        if any(keyword in line for keyword in keywords):
            matched.append((n, line.strip()))

    for n, line in matched[:65]:
        print(f"{n:4d}: {line[:180]}")

    if len(matched) > 65:
        print(f"... 나머지 {len(matched)-65}개 행 생략")

print("\n===== 8단계 확인 완료 =====")

===== 8단계. Moment-DETR 원본 후보 추출 함수 확인 =====

함수: predict_moments
시작 행: 386
종료 행: 567
인자: ['video_id', 'video_feature_path', 'query', 'video_duration', 'top_k', 'iou_threshold']

[함수 시작 부분]
 386: def predict_moments(
 387:     video_id,
 388:     video_feature_path,
 389:     query,
 390:     video_duration=None,
 391:     top_k=5,
 392:     iou_threshold=0.5
 393: ):
 394:     """
 395:     Moment-DETR의 서로 다른 구간 Top-K 반환.
 396:     """
 397: 
 398:     # ----------------------------------------------
 399:     # Video feature
 400:     # ----------------------------------------------
 401: 
 402:     video_feats = torch.load(
 403:         video_feature_path,
 404:         map_location=DEVICE
 405:     ).float()
 406: 
 407:     video_feats = F.normalize(
 408:         video_feats,
 409:         dim=-1,
 410:         eps=1e-5
 411:     )
 412: 
 413:     n_frames = len(video_feats)
 414: 
 415:     tef_st = (
 416:         torch.arange(
 417:             n_frames,
 418:             dty

In [10]:

from pathlib import Path
import importlib.util

BASE_DIR = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "04_PinPoint_ConsensusV3_MultilingualCLIP"
)

CODE_DIR = BASE_DIR / "01_code"
CODE_DIR.mkdir(parents=True, exist_ok=True)

code = r'''
def temporal_iou(a, b):
    start = max(float(a["start_time"]), float(b["start_time"]))
    end = min(float(a["end_time"]), float(b["end_time"]))

    intersection = max(0.0, end - start)

    duration_a = float(a["end_time"]) - float(a["start_time"])
    duration_b = float(b["end_time"]) - float(b["start_time"])

    union = duration_a + duration_b - intersection

    return intersection / union if union > 0 else 0.0


def rank_by_consensus_v3(visual, ko, en, overlap_threshold=0.5):
    """
    Consensus V3 실험용.

    입력:
        visual: Moment-DETR 후보
        ko: MiniLM 한국어 후보
        en: MiniLM 영어 후보

    출력:
        consensus_score 내림차순 후보 목록

    Ground Truth는 사용하지 않는다.
    """

    groups = {
        "visual": visual[:3],
        "ko": ko[:3],
        "en": en[:3],
    }

    candidates = []

    for source, group in groups.items():
        for rank, candidate in enumerate(group, 1):
            item = dict(candidate)

            item["rank_source"] = source
            item["original_rank"] = rank
            item["rank_score"] = 1.0 / rank

            candidates.append(item)

    for candidate in candidates:
        source = candidate["rank_source"]

        supports = {}

        for other_source, group in groups.items():
            if other_source == source:
                continue

            best_iou = max(
                (
                    temporal_iou(candidate, other)
                    for other in group
                ),
                default=0.0
            )

            supports[other_source] = best_iou

        matching_sources = [
            s for s, overlap in supports.items()
            if overlap >= overlap_threshold
        ]

        bonus = 0.0

        # Visual과 자막 검색이 함께 동의
        if source == "visual":
            if "ko" in matching_sources or "en" in matching_sources:
                bonus += 0.6

            if "ko" in matching_sources and "en" in matching_sources:
                bonus += 0.2

        # 자막 후보에 Visual이 동의
        else:
            if "visual" in matching_sources:
                bonus += 0.6

            # KO와 EN은 같은 MiniLM이므로 가산점 축소
            other_text_source = "en" if source == "ko" else "ko"

            if other_text_source in matching_sources:
                bonus += 0.2

        candidate["support_iou"] = supports
        candidate["support_sources"] = matching_sources
        candidate["agreement_bonus"] = bonus

        candidate["consensus_score"] = (
            candidate["rank_score"] + bonus
        )

    return sorted(
        candidates,
        key=lambda x: x["consensus_score"],
        reverse=True
    )
'''

file_path = CODE_DIR / "consensus_v3.py"

if file_path.exists():
    raise FileExistsError(
        f"기존 파일을 보호하기 위해 덮어쓰지 않습니다: {file_path}"
    )

file_path.write_text(code.strip() + "\n", encoding="utf-8")

print("===== 9단계. Consensus V3 코드 저장 =====")
print("저장 위치:", file_path)

spec = importlib.util.spec_from_file_location(
    "consensus_v3_experiment",
    file_path
)

module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

visual = [
    {"start_time": 10, "end_time": 20, "score": 0.8},
    {"start_time": 40, "end_time": 50, "score": 0.7},
    {"start_time": 60, "end_time": 70, "score": 0.6},
]

ko = [
    {"start_time": 11, "end_time": 21, "score": 0.7},
    {"start_time": 80, "end_time": 90, "score": 0.6},
    {"start_time": 100, "end_time": 110, "score": 0.5},
]

en = [
    {"start_time": 12, "end_time": 22, "score": 0.7},
    {"start_time": 120, "end_time": 130, "score": 0.6},
    {"start_time": 140, "end_time": 150, "score": 0.5},
]

results = module.rank_by_consensus_v3(visual, ko, en)

print("\n===== 가상 후보 테스트 =====")

for rank, candidate in enumerate(results[:5], 1):
    print(
        f"{rank}위 | "
        f"{candidate['rank_source']} | "
        f"{candidate['start_time']}~{candidate['end_time']}초 | "
        f"점수={candidate['consensus_score']:.4f} | "
        f"동의={candidate['support_sources']}"
    )

print("\n===== 9단계 완료 =====")

===== 9단계. Consensus V3 코드 저장 =====
저장 위치: /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/01_code/consensus_v3.py

===== 가상 후보 테스트 =====
1위 | visual | 10~20초 | 점수=1.8000 | 동의=['ko', 'en']
2위 | ko | 11~21초 | 점수=1.8000 | 동의=['visual', 'en']
3위 | en | 12~22초 | 점수=1.8000 | 동의=['visual', 'ko']
4위 | visual | 40~50초 | 점수=0.5000 | 동의=[]
5위 | ko | 80~90초 | 점수=0.5000 | 동의=[]

===== 9단계 완료 =====


In [11]:

from pathlib import Path

print("===== 10단계. 실제 모델 및 데이터 경로 확인 =====")

DRIVE = Path("/content/drive/MyDrive/pinpoint_capstone")

CHECK_DIRS = [
    DRIVE / "01_AI_인수인계",
    DRIVE / "02_Korean_Video_Test/clip_features",
    DRIVE / "02_Korean_Video_Test/AI_Backup",
    DRIVE / "03_Korean_Training_Data",
    DRIVE / "04_PinPoint_ConsensusV3_MultilingualCLIP",
]

for folder in CHECK_DIRS:
    print("\n" + "=" * 65)
    print("폴더:", folder)
    print("존재:", folder.exists())

    if not folder.exists():
        continue

    for item in sorted(folder.iterdir(), key=lambda x: x.name):
        if item.is_file():
            size = item.stat().st_size / (1024 ** 2)
            print(f"[파일] {item.name} ({size:.2f} MB)")
        else:
            print(f"[폴더] {item.name}")

print("\n===== 주요 모델 파일 탐색 =====")

MODEL_EXTENSIONS = {
    ".ckpt", ".safetensors", ".pth", ".pt"
}

for root_name in [
    "01_AI_인수인계",
    "02_Korean_Video_Test/AI_Backup",
    "03_Korean_Training_Data"
]:
    root = DRIVE / root_name

    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in MODEL_EXTENSIONS:
            size = path.stat().st_size / (1024 ** 2)

            print(
                f"{path.relative_to(DRIVE)} "
                f"({size:.2f} MB)"
            )

print("\n===== 10단계 완료 =====")

===== 10단계. 실제 모델 및 데이터 경로 확인 =====

폴더: /content/drive/MyDrive/pinpoint_capstone/01_AI_인수인계
존재: True
[폴더] 01_문서
[폴더] 02_테스트_원본
[폴더] 03_전처리_결과
[폴더] 04_MomentDETR_외부코드
[폴더] 05_생성_Features
[폴더] 06_안내

폴더: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/clip_features
존재: True
[파일] kr_daiso_001_clip_features.pt (0.03 MB)
[파일] kr_kitchen_001_clip_features.pt (0.04 MB)
[파일] kr_tofu_001_clip_features.pt (0.16 MB)

폴더: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup
존재: True
[파일] KOREAN_AI_INTEGRATION_restored_20261010.md (0.01 MB)
[파일] consensus_rerank.py (0.00 MB)
[파일] fusion_search_20261010.py (0.01 MB)
[파일] fusion_search_FINAL_restored_20261010.py (0.01 MB)
[파일] fusion_search_before_model_connection_20261010_130236.py (0.01 MB)
[파일] fusion_search_before_model_selection_20261010_130203.py (0.01 MB)
[파일] minilm_README_restored_20261010.md (0.01 MB)
[파일] minilm_evaluate_restored_20261010.py (0.00 MB)
[파일] minilm_full_validation

In [12]:

from pathlib import Path

print("===== 11단계. 모델 및 Feature 구조 확인 =====")

DRIVE = Path("/content/drive/MyDrive/pinpoint_capstone")

print("\n[1] CLIP 특징 파일 전체 확인")

feature_files = sorted(
    p for p in DRIVE.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".pt", ".npz"}
    and ("clip" in p.name.lower() or "squat" in str(p).lower())
)

for path in feature_files:
    print(path.relative_to(DRIVE))

print("특징 파일 수:", len(feature_files))

print("\n[2] Fine-tuned MiniLM 최종 모델 구조")

MODEL_DIR = (
    DRIVE
    / "03_Korean_Training_Data"
    / "minilm_finetuned"
    / "final_model"
)

print("모델 폴더 존재:", MODEL_DIR.is_dir())

if MODEL_DIR.is_dir():
    for path in sorted(MODEL_DIR.rglob("*")):
        if not path.is_file():
            continue

        size_mb = path.stat().st_size / (1024 ** 2)
        print(
            f"{path.relative_to(MODEL_DIR)} "
            f"({size_mb:.2f} MB)"
        )

print("\n[3] Moment-DETR 체크포인트")

CKPT = (
    DRIVE
    / "02_Korean_Video_Test"
    / "AI_Backup"
    / "scratch_clip_align_best_ORIGINAL.ckpt"
)

print("체크포인트 존재:", CKPT.is_file())

if CKPT.is_file():
    print(
        "크기:",
        round(CKPT.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

print("\n===== 11단계 완료 =====")

===== 11단계. 모델 및 Feature 구조 확인 =====

[1] CLIP 특징 파일 전체 확인
01_AI_인수인계/05_생성_Features/video_001_clip_features.pt
02_Korean_Video_Test/clip_features/kr_daiso_001_clip_features.pt
02_Korean_Video_Test/clip_features/kr_kitchen_001_clip_features.pt
02_Korean_Video_Test/clip_features/kr_tofu_001_clip_features.pt
특징 파일 수: 4

[2] Fine-tuned MiniLM 최종 모델 구조
모델 폴더 존재: True
1_Pooling/config.json (0.00 MB)
README.md (0.02 MB)
config.json (0.00 MB)
config_sentence_transformers.json (0.00 MB)
model.safetensors (448.83 MB)
modules.json (0.00 MB)
sentence_bert_config.json (0.00 MB)
special_tokens_map.json (0.00 MB)
tokenizer.json (16.29 MB)
tokenizer_config.json (0.00 MB)

[3] Moment-DETR 체크포인트
체크포인트 존재: True
크기: 48.88 MB

===== 11단계 완료 =====


In [13]:

from pathlib import Path
import subprocess

print("===== 12단계. PinPoint GitHub 코드 가져오기 =====")

REPO_URL = "https://github.com/pinpoint-capstone/pinpoint-capstone.git"
TARGET = Path("/content/pinpoint-capstone")

if TARGET.exists():
    print("기존 폴더가 있으므로 덮어쓰지 않습니다.")
else:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(TARGET)],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("✅ GitHub Clone 성공")
    else:
        print("❌ Clone 실패")
        print(result.stderr[-1500:])

if (TARGET / ".git").exists():
    print("\n===== 현재 GitHub 코드 상태 =====")

    commands = [
        ["git", "-C", str(TARGET), "branch", "--show-current"],
        ["git", "-C", str(TARGET), "log", "-1", "--oneline"],
        ["git", "-C", str(TARGET), "status", "--short"],
    ]

    for cmd in commands:
        result = subprocess.run(
            cmd, capture_output=True, text=True
        )
        print(" ".join(cmd[-2:]), ":", result.stdout.strip())

    print("\n===== 주요 AI 파일 확인 =====")

    files = [
        "ai/features/run_moment_detr_inference.py",
        "ai/features/fusion_search.py",
        "ai/features/consensus_rerank.py",
    ]

    for name in files:
        path = TARGET / name
        print("✅" if path.exists() else "❌", name)

print("\n===== 12단계 완료 =====")

===== 12단계. PinPoint GitHub 코드 가져오기 =====
✅ GitHub Clone 성공

===== 현재 GitHub 코드 상태 =====
branch --show-current : main
-1 --oneline : 14b3eb9 fix: include CLIP tokenizer vocabulary
status --short : 

===== 주요 AI 파일 확인 =====
✅ ai/features/run_moment_detr_inference.py
❌ ai/features/fusion_search.py
❌ ai/features/consensus_rerank.py

===== 12단계 완료 =====


In [14]:

import subprocess

REPO = "/content/pinpoint-capstone"

print("===== 13단계. GitHub 원격 브랜치 확인 =====")

result = subprocess.run(
    ["git", "-C", REPO, "ls-remote", "--heads", "origin"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("브랜치 조회 실패")
    print(result.stderr)
else:
    branches = []

    for line in result.stdout.splitlines():
        commit, ref = line.split("\t")
        branch = ref.replace("refs/heads/", "")
        branches.append((branch, commit[:10]))

    print("원격 브랜치 수:", len(branches))

    for branch, commit in branches:
        print(f"브랜치: {branch} | 커밋: {commit}")

print("\n===== 13단계 완료 =====")

===== 13단계. GitHub 원격 브랜치 확인 =====
원격 브랜치 수: 4
브랜치: feature/backend | 커밋: 876fc4c3a8
브랜치: feature/frontend | 커밋: 5b39d1f857
브랜치: feature/korean-ai-fusion | 커밋: 5b7df89251
브랜치: main | 커밋: 14b3eb9f6f

===== 13단계 완료 =====


In [15]:

import subprocess
from pathlib import Path

print("===== 14단계. AI 브랜치 코드 확인 =====")

REPO = Path("/content/pinpoint-capstone")
BRANCH = "feature/korean-ai-fusion"

if not (REPO / ".git").exists():
    raise FileNotFoundError("GitHub 저장소 폴더가 없습니다.")

print("\n[1] AI 브랜치 가져오기")

fetch = subprocess.run(
    [
        "git", "-C", str(REPO),
        "fetch", "origin",
        f"{BRANCH}:refs/remotes/origin/{BRANCH}"
    ],
    capture_output=True,
    text=True
)

print("Fetch 성공:", fetch.returncode == 0)

if fetch.returncode != 0:
    print(fetch.stderr[-2000:])
    raise RuntimeError("브랜치 가져오기에 실패했습니다.")

print("\n[2] AI 브랜치 최신 커밋")

commit = subprocess.run(
    [
        "git", "-C", str(REPO),
        "log", "-1", "--oneline",
        f"origin/{BRANCH}"
    ],
    capture_output=True,
    text=True,
    check=True
)

print(commit.stdout.strip())

print("\n[3] AI 관련 파일 확인")

files = subprocess.run(
    [
        "git", "-C", str(REPO),
        "ls-tree", "-r", "--name-only",
        f"origin/{BRANCH}"
    ],
    capture_output=True,
    text=True,
    check=True
)

paths = files.stdout.splitlines()

keywords = [
    "fusion",
    "consensus",
    "moment_detr",
    "minilm",
    "transcribe",
]

for path in paths:
    if (
        path.startswith("ai/")
        and any(k in path.lower() for k in keywords)
    ):
        print(path)

print("\n[4] 현재 작업 브랜치 확인")

current = subprocess.run(
    ["git", "-C", str(REPO), "branch", "--show-current"],
    capture_output=True,
    text=True,
    check=True
)

print("현재 브랜치:", current.stdout.strip())

print("\n===== 14단계 완료 =====")

===== 14단계. AI 브랜치 코드 확인 =====

[1] AI 브랜치 가져오기
Fetch 성공: True

[2] AI 브랜치 최신 커밋
5b7df89 docs: add Korean video AI experiment results

[3] AI 관련 파일 확인
ai/experiments/korean_minilm/README.md
ai/experiments/korean_minilm/evaluate.py
ai/experiments/korean_minilm/prepare_data.py
ai/experiments/korean_minilm/results/full_validation_metrics.json
ai/experiments/korean_minilm/train.py
ai/experiments/korean_video/results/consensus_6queries_comparison.json
ai/experiments/korean_video/results/consensus_v2_12queries_comparison.json
ai/experiments/korean_video/results/fusion_baseline_evaluation.json
ai/experiments/korean_video/results/fusion_new_3queries_comparison.json
ai/experiments/korean_video/results/fusion_start_time_comparison.json
ai/experiments/korean_video/results/minilm_baseline_evaluation.json
ai/experiments/korean_video/results/minilm_finetuning_comparison.json
ai/experiments/korean_video/results/minilm_full_validation_comparison.json
ai/experiments/korean_video/results/minilm_full_val

In [16]:

from pathlib import Path
import subprocess

print("===== 15단계. AI 브랜치 별도 작업 폴더 생성 =====")

MAIN_REPO = Path("/content/pinpoint-capstone")
AI_REPO = Path("/content/pinpoint-capstone-ai")
BRANCH = "feature/korean-ai-fusion"

if not (MAIN_REPO / ".git").exists():
    raise FileNotFoundError("기존 GitHub 저장소가 없습니다.")

if AI_REPO.exists():
    print("AI 작업 폴더가 이미 존재하므로 덮어쓰지 않습니다.")

else:
    result = subprocess.run(
        [
            "git", "-C", str(MAIN_REPO),
            "worktree", "add",
            "--detach",
            str(AI_REPO),
            f"origin/{BRANCH}"
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("AI 브랜치 작업 폴더 생성 실패")

    print("✅ AI 브랜치 작업 폴더 생성 완료")

print("\n===== AI 작업 폴더 상태 =====")

if AI_REPO.exists():
    for command in [
        ["git", "-C", str(AI_REPO), "log", "-1", "--oneline"],
        ["git", "-C", str(AI_REPO), "status", "--short"],
    ]:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True
        )
        print("명령:", " ".join(command[-3:]))
        print("결과:", result.stdout.strip() or "(변경 없음)")

    print("\n===== 핵심 파일 확인 =====")

    files = [
        "ai/features/fusion_search.py",
        "ai/features/consensus_rerank.py",
        "ai/features/run_moment_detr_inference.py",
        "ai/features/load_moment_detr_model.py",
    ]

    for name in files:
        path = AI_REPO / name
        print("✅" if path.is_file() else "❌", name)

print("\n===== 기존 main 유지 확인 =====")

result = subprocess.run(
    [
        "git", "-C", str(MAIN_REPO),
        "branch", "--show-current"
    ],
    capture_output=True,
    text=True
)

print("main 작업 폴더 브랜치:", result.stdout.strip())

print("\n===== 15단계 완료 =====")

===== 15단계. AI 브랜치 별도 작업 폴더 생성 =====
✅ AI 브랜치 작업 폴더 생성 완료

===== AI 작업 폴더 상태 =====
명령: log -1 --oneline
결과: 5b7df89 docs: add Korean video AI experiment results
명령: /content/pinpoint-capstone-ai status --short
결과: (변경 없음)

===== 핵심 파일 확인 =====
✅ ai/features/fusion_search.py
✅ ai/features/consensus_rerank.py
✅ ai/features/run_moment_detr_inference.py
✅ ai/features/load_moment_detr_model.py

===== 기존 main 유지 확인 =====
main 작업 폴더 브랜치: main

===== 15단계 완료 =====


In [17]:

from pathlib import Path
import ast

print("===== 16단계. AI 브랜치 모델·입력 설정 확인 =====")

ROOT = Path("/content/pinpoint-capstone-ai")

TARGETS = [
    "ai/features/run_moment_detr_inference.py",
    "ai/features/load_moment_detr_model.py",
    "ai/features/fusion_search.py",
]

KEYWORDS = [
    "CHECKPOINT",
    "CKPT",
    "MODEL_PATH",
    "PINPOINT_",
    "clip.load",
    "torch.load",
    "video_feature_path",
    "feature_path",
    "get_model",
    "predict_moments",
    "SentenceTransformer",
    "get_semantic_model",
    "DEVICE",
]

for relative_path in TARGETS:
    path = ROOT / relative_path

    print("\n" + "=" * 70)
    print("파일:", relative_path)
    print("존재:", path.exists())

    if not path.exists():
        continue

    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()

    try:
        tree = ast.parse(source)
        print("문법 분석: 정상")
    except SyntaxError as e:
        print("문법 오류:", e)
        continue

    print("전체 줄 수:", len(lines))

    print("\n[주요 함수 및 인자]")

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if any(
                word in node.name
                for word in ["model", "predict", "feature", "clip"]
            ):
                args = [arg.arg for arg in node.args.args]
                print(
                    f"{node.lineno}: "
                    f"{node.name}({', '.join(args)})"
                )

    print("\n[모델 및 입력 관련 코드]")

    count = 0

    for number, line in enumerate(lines, 1):
        if any(keyword in line for keyword in KEYWORDS):
            print(f"{number}: {line.strip()[:180]}")
            count += 1

            if count >= 45:
                print("... 출력 제한")
                break

print("\n===== 16단계 완료 =====")

===== 16단계. AI 브랜치 모델·입력 설정 확인 =====

파일: ai/features/run_moment_detr_inference.py
존재: True
문법 분석: 정상
전체 줄 수: 1414

[주요 함수 및 인자]
52: get_model()
68: get_feature_extractor()
89: predict_moment(video_id, video_feature_path, query, video_duration)
386: predict_moments(video_id, video_feature_path, query, video_duration, top_k, iou_threshold)
588: get_refine_clip_model()
938: predict_moment_hybrid(video_id, video_feature_path, query, subtitle_json_path, video_duration)
1232: predict_moments_hybrid(video_id, video_feature_path, query, subtitle_json_path, video_duration, top_k, iou_threshold, min_start_gap)

[모델 및 입력 관련 코드]
32: DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
35: CHECKPOINT_PATH = (
52: def get_model():
56: print(f"[AI] Loading Moment-DETR on {DEVICE}")
59: str(CHECKPOINT_PATH)
62: _MODEL = _MODEL.to(DEVICE)
72: print(f"[AI] Loading CLIP on {DEVICE}")
79: device=DEVICE
91: video_feature_path,
103: video_feature_path : str | Path
129: video_feature_path = Path(video_fe

In [18]:

from pathlib import Path
import ast
import torch

print("===== 17단계. Moment-DETR 모델·입력 호환성 확인 =====")

REPO = Path("/content/pinpoint-capstone-ai")
CODE = REPO / "ai/features/run_moment_detr_inference.py"

lines = CODE.read_text(encoding="utf-8").splitlines()

print("\n[1] 모델 로딩 및 경로 설정 코드")

for start, end in [(1, 90)]:
    for n in range(start, min(end, len(lines)) + 1):
        print(f"{n:4d}: {lines[n-1]}")

print("\n[2] Visual Top-K 함수의 핵심 처리 코드")

for start, end in [
    (386, 410),
    (425, 510),
    (530, 567),
]:
    print(f"\n--- {start}~{end}행 ---")
    for n in range(start, min(end, len(lines)) + 1):
        print(f"{n:4d}: {lines[n-1]}")

print("\n[3] 기존 한국어 영상 CLIP 특징 검사")

FEATURE_DIR = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/clip_features"
)

for path in sorted(FEATURE_DIR.glob("*.pt")):
    print(f"\n파일: {path.name}")

    try:
        data = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        print("자료형:", type(data).__name__)

        if isinstance(data, torch.Tensor):
            print("Shape:", tuple(data.shape))
            print("Dtype:", data.dtype)
            print("NaN 포함:", bool(torch.isnan(data).any()))
            print("Inf 포함:", bool(torch.isinf(data).any()))

        elif isinstance(data, dict):
            print("키:", list(data.keys()))
            for key, value in data.items():
                if isinstance(value, torch.Tensor):
                    print(
                        f"  {key}: shape={tuple(value.shape)}"
                    )

    except Exception as e:
        print("읽기 오류:", type(e).__name__, str(e)[:300])

print("\n===== 17단계 완료 =====")

===== 17단계. Moment-DETR 모델·입력 호환성 확인 =====

[1] 모델 로딩 및 경로 설정 코드
   1: from pathlib import Path
   2: import sys
   3: 
   4: import torch
   5: import torch.nn.functional as F
   6: 
   7: 
   8: # ==================================================
   9: # Path 설정
  10: # ==================================================
  11: 
  12: PROJECT_ROOT = Path(__file__).resolve().parents[2]
  13: MOMENT_DETR_ROOT = PROJECT_ROOT / "ai" / "vendor" / "moment_detr"
  14: 
  15: sys.path.append(str(MOMENT_DETR_ROOT))
  16: 
  17: 
  18: # ==================================================
  19: # Moment-DETR import
  20: # ==================================================
  21: 
  22: from run_on_video.data_utils import ClipFeatureExtractor
  23: from run_on_video.model_utils import build_inference_model
  24: from utils.tensor_utils import pad_sequences_1d
  25: from moment_detr.span_utils import span_cxw_to_xx
  26: 
  27: 
  28: # ==================================================
  29: # 공통

In [19]:

from pathlib import Path
import sys
import importlib
import importlib.util
import torch

print("===== 18단계. 실제 Moment-DETR 실행 준비 =====")

REPO = Path("/content/pinpoint-capstone-ai")

CKPT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "scratch_clip_align_best_ORIGINAL.ckpt"
)

FEATURE_DIR = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/clip_features"
)

print("\n[1] 입력 파일 확인")

print("AI 브랜치:", REPO.exists())
print("체크포인트:", CKPT.is_file())
print("체크포인트 크기(MB):",
      round(CKPT.stat().st_size / 1024**2, 2)
      if CKPT.is_file() else "없음")

for name in [
    "kr_tofu_001",
    "kr_kitchen_001",
    "kr_daiso_001",
]:
    path = FEATURE_DIR / f"{name}_clip_features.pt"
    print(f"{name} 특징:", path.is_file())

print("\n[2] 필수 라이브러리 확인")

packages = [
    "torch",
    "numpy",
    "clip",
    "sentence_transformers",
    "langdetect",
    "easydict",
    "transformers",
]

for package in packages:
    try:
        spec = importlib.util.find_spec(package)
        print(
            "✅" if spec is not None else "❌",
            package
        )
    except (ImportError, ValueError, ModuleNotFoundError) as e:
        print("❌", package, type(e).__name__)

print("\n[3] GitHub AI 코드 import 준비")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

module_name = "ai.features.run_moment_detr_inference"

try:
    inference = importlib.import_module(module_name)

    # 원본 GitHub 파일은 수정하지 않고
    # 현재 Python 세션의 모델 경로만 설정
    inference.CHECKPOINT_PATH = CKPT

    print("✅ Moment-DETR 추론 모듈 import 성공")
    print("실제 사용 체크포인트:", inference.CHECKPOINT_PATH)
    print("GPU:", inference.DEVICE)

except Exception as e:
    inference = None
    print("❌ 추론 모듈 import 실패")
    print("오류 종류:", type(e).__name__)
    print("오류 내용:", str(e)[:1000])

print("\n===== 18단계 완료 =====")

===== 18단계. 실제 Moment-DETR 실행 준비 =====

[1] 입력 파일 확인
AI 브랜치: True
체크포인트: True
체크포인트 크기(MB): 48.88
kr_tofu_001 특징: True
kr_kitchen_001 특징: True
kr_daiso_001 특징: True

[2] 필수 라이브러리 확인
✅ torch
✅ numpy
❌ clip
✅ sentence_transformers
❌ langdetect
✅ easydict
✅ transformers

[3] GitHub AI 코드 import 준비
❌ 추론 모듈 import 실패
오류 종류: ModuleNotFoundError
오류 내용: No module named 'ffmpeg'

===== 18단계 완료 =====


In [20]:

print("===== 19단계. 누락된 라이브러리 설치 =====")

%pip install -q ffmpeg-python langdetect

print("\n===== 설치 후 확인 =====")

import ffmpeg
import langdetect

print("✅ ffmpeg-python import 성공")
print("✅ langdetect import 성공")

print("\n===== 19단계 완료 =====")

===== 19단계. 누락된 라이브러리 설치 =====
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

===== 설치 후 확인 =====
✅ ffmpeg-python import 성공
✅ langdetect import 성공

===== 19단계 완료 =====


In [22]:

print("===== 21단계. ftfy 설치 및 확인 =====")

%pip install -q ftfy

print("\n===== 설치 확인 =====")

import ftfy

print("✅ ftfy import 성공")
print("설치 위치:", ftfy.__file__)

print("\n===== 21단계 완료 =====")

===== 21단계. ftfy 설치 및 확인 =====
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00

===== 설치 확인 =====
✅ ftfy import 성공
설치 위치: /usr/local/lib/python3.13/dist-packages/ftfy/__init__.py

===== 21단계 완료 =====


In [23]:

from pathlib import Path
import sys
import importlib
import traceback

print("===== 22단계. Moment-DETR 추론 모듈 연결 =====")

REPO = Path("/content/pinpoint-capstone-ai")

VENDOR = REPO / "ai/vendor/moment_detr"

CKPT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/AI_Backup/"
    "scratch_clip_align_best_ORIGINAL.ckpt"
)

print("\n[1] 필수 경로 확인")

for name, path in [
    ("AI 브랜치", REPO),
    ("Moment-DETR 코드", VENDOR),
    ("학습 체크포인트", CKPT),
]:
    print(
        "✅" if path.exists() else "❌",
        name,
        path
    )

if not all(p.exists() for p in [REPO, VENDOR, CKPT]):
    raise FileNotFoundError("필수 파일 또는 폴더가 없습니다.")

print("\n[2] Python 경로 설정")

for path in [str(REPO), str(VENDOR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("\n[3] 프로젝트 내부 CLIP 확인")

try:
    clip_module = importlib.import_module(
        "run_on_video.clip.clip"
    )
    print("✅ CLIP import 성공")
    print("위치:", clip_module.__file__)
except Exception as e:
    print("❌ CLIP import 실패:", type(e).__name__, str(e))

print("\n[4] Moment-DETR 추론 모듈 확인")

try:
    inference = importlib.import_module(
        "ai.features.run_moment_detr_inference"
    )

    inference.CHECKPOINT_PATH = CKPT

    print("✅ Moment-DETR import 성공")
    print("추론 코드:", inference.__file__)
    print("연결된 체크포인트:", inference.CHECKPOINT_PATH)
    print("사용 장치:", inference.DEVICE)
    print(
        "Visual Top-3 함수 사용 가능:",
        callable(getattr(inference, "predict_moments", None))
    )

except Exception as e:
    inference = None
    print("❌ Moment-DETR import 실패")
    print("오류 종류:", type(e).__name__)
    print("오류 내용:", str(e))
    traceback.print_exc(limit=5)

print("\n===== 22단계 완료 =====")

===== 22단계. Moment-DETR 추론 모듈 연결 =====

[1] 필수 경로 확인
✅ AI 브랜치 /content/pinpoint-capstone-ai
✅ Moment-DETR 코드 /content/pinpoint-capstone-ai/ai/vendor/moment_detr
✅ 학습 체크포인트 /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/scratch_clip_align_best_ORIGINAL.ckpt

[2] Python 경로 설정

[3] 프로젝트 내부 CLIP 확인
✅ CLIP import 성공
위치: /content/pinpoint-capstone-ai/ai/vendor/moment_detr/run_on_video/clip/clip.py

[4] Moment-DETR 추론 모듈 확인
✅ Moment-DETR import 성공
추론 코드: /content/pinpoint-capstone-ai/ai/features/run_moment_detr_inference.py
연결된 체크포인트: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/scratch_clip_align_best_ORIGINAL.ckpt
사용 장치: cuda
Visual Top-3 함수 사용 가능: True

===== 22단계 완료 =====


In [24]:

import torch
import traceback

print("===== 23단계. Moment-DETR 실제 모델 로딩 =====")

try:
    # 22단계에서 import한 inference 모듈 사용
    assert inference is not None, "추론 모듈이 연결되지 않았습니다."

    print("\n[1] 체크포인트")
    print(inference.CHECKPOINT_PATH)

    print("\n[2] 모델 로딩 시작")

    model = inference.get_model()

    print("\n✅ Moment-DETR 모델 로딩 성공")
    print("모델 클래스:", type(model).__name__)
    print("평가 모드:", not model.training)

    parameters = sum(p.numel() for p in model.parameters())
    print("총 파라미터 수:", f"{parameters:,}")

    first_parameter = next(model.parameters(), None)

    if first_parameter is not None:
        print("모델 장치:", first_parameter.device)

    print("\n[3] GPU 메모리")

    if torch.cuda.is_available():
        print(
            "할당된 GPU 메모리:",
            round(
                torch.cuda.memory_allocated() / 1024**3,
                3
            ),
            "GB"
        )

except Exception as e:
    print("\n❌ Moment-DETR 모델 로딩 실패")
    print("오류 종류:", type(e).__name__)
    print("오류 내용:", str(e))
    traceback.print_exc(limit=8)

print("\n===== 23단계 완료 =====")

===== 23단계. Moment-DETR 실제 모델 로딩 =====

[1] 체크포인트
/content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/scratch_clip_align_best_ORIGINAL.ckpt

[2] 모델 로딩 시작
[AI] Loading Moment-DETR on cuda

✅ Moment-DETR 모델 로딩 성공
모델 클래스: MomentDETR
평가 모드: True
총 파라미터 수: 4,278,217
모델 장치: cuda:0

[3] GPU 메모리
할당된 GPU 메모리: 0.016 GB

===== 23단계 완료 =====


In [25]:

from pathlib import Path
import json
import traceback

print("===== 24단계. 두부 영상 Moment-DETR Visual Top-3 =====")

ROOT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test"
)

FEATURE = ROOT / "clip_features/kr_tofu_001_clip_features.pt"

EVAL = ROOT / "fusion_baseline_evaluation.json"

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "04_PinPoint_ConsensusV3_MultilingualCLIP/"
    "02_results"
)

print("\n[1] 기존 평가 데이터 확인")

with EVAL.open("r", encoding="utf-8") as f:
    evaluation = json.load(f)

tofu_gt = evaluation["ground_truth"]["두부"]

print(json.dumps(tofu_gt, ensure_ascii=False, indent=2))

print("\n[2] 기존 영어 검색어 확인")

if not isinstance(tofu_gt, dict):
    raise TypeError("두부 정답 정보 형식이 예상과 다릅니다.")

query_en = tofu_gt.get("query_en")

if not query_en:
    print("영어 검색어가 ground_truth에 없습니다.")
    print("기존 results 구조를 추가 확인해야 합니다.")

else:
    print("영어 검색어:", query_en)

    print("\n[3] Visual Top-3 검색")

    try:
        # 22~23단계에서 연결하고 로딩한 모델 사용
        predictions = inference.predict_moments(
            video_id="kr_tofu_001",
            video_feature_path=str(FEATURE),
            query=query_en,
            video_duration=None,
            top_k=3
        )

        print("반환 후보 수:", len(predictions))

        for rank, candidate in enumerate(predictions, 1):
            print(
                f"{rank}위 | "
                f"{candidate['start_time']}~"
                f"{candidate['end_time']}초 | "
                f"score={candidate['score']:.4f}"
            )

        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

        output_path = (
            OUTPUT_DIR / "tofu_visual_top3_step24.json"
        )

        if output_path.exists():
            raise FileExistsError(
                f"기존 결과를 덮어쓰지 않습니다: {output_path}"
            )

        with output_path.open("w", encoding="utf-8") as f:
            json.dump(
                {
                    "video_id": "kr_tofu_001",
                    "query_en": query_en,
                    "method": "Moment-DETR Visual-only",
                    "top_k": 3,
                    "video_duration_source": "feature_length_x_2s",
                    "predictions": predictions
                },
                f,
                ensure_ascii=False,
                indent=2
            )

        print("\n✅ 결과 저장 완료")
        print(output_path)

    except Exception as e:
        print("\n❌ 추론 또는 결과 저장 실패")
        print("오류:", type(e).__name__, str(e))
        traceback.print_exc(limit=6)

print("\n===== 24단계 완료 =====")

===== 24단계. 두부 영상 Moment-DETR Visual Top-3 =====

[1] 기존 평가 데이터 확인
{
  "start_time": 12.0,
  "end_time": 23.0
}

[2] 기존 영어 검색어 확인
영어 검색어가 ground_truth에 없습니다.
기존 results 구조를 추가 확인해야 합니다.

===== 24단계 완료 =====


In [26]:

from pathlib import Path
import json

print("===== 24-1단계. 기존 두부 검색어 확인 =====")

ROOT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test"
)

files = [
    ROOT / "fusion_baseline_evaluation.json",
    ROOT / "AI_Backup/public_experiment_results_20261010/"
           "consensus_6queries_comparison.json",
    ROOT / "AI_Backup/public_experiment_results_20261010/"
           "minilm_fusion_6queries_comparison.json",
]

for path in files:
    print("\n" + "=" * 65)
    print("파일:", path.name)

    if not path.exists():
        print("❌ 파일 없음")
        continue

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if path.name == "fusion_baseline_evaluation.json":
        print("\n[두부 관련 평가 정보]")
        print(json.dumps(
            data.get("results", {}).get("두부"),
            ensure_ascii=False,
            indent=2
        )[:5000])

    else:
        print("\n[검색 사례 이름 및 Query 확인]")

        for item in data.get("results", []):
            print("사례:", item.get("name"))
            print("영상:", item.get("video_id"))

            for key in ["query", "query_ko", "query_en"]:
                if key in item:
                    print(key, ":", item[key])

            candidates = item.get(
                "predictions",
                item.get("candidates", [])
            )

            if candidates:
                first = candidates[0]
                print(
                    "첫 후보 query:",
                    first.get("query", "없음")
                )

            print("-" * 30)

print("\n===== 24-1단계 완료 =====")

===== 24-1단계. 기존 두부 검색어 확인 =====

파일: fusion_baseline_evaluation.json

[두부 관련 평가 정보]
[
  {
    "rank": 1,
    "source": "hybrid",
    "language": null,
    "start_time": 98.5,
    "end_time": 118.9725,
    "iou": 0.0
  },
  {
    "rank": 2,
    "source": "subtitle",
    "language": "ko",
    "start_time": 140.86,
    "end_time": 143.58,
    "iou": 0.0
  },
  {
    "rank": 3,
    "source": "subtitle",
    "language": "en",
    "start_time": 12.88,
    "end_time": 21.06,
    "iou": 0.7436363636363634
  },
  {
    "rank": 4,
    "source": "hybrid",
    "language": null,
    "start_time": 84.5,
    "end_time": 102.7907,
    "iou": 0.0
  },
  {
    "rank": 5,
    "source": "subtitle",
    "language": "ko",
    "start_time": 135.06,
    "end_time": 137.58,
    "iou": 0.0
  }
]

파일: consensus_6queries_comparison.json

[검색 사례 이름 및 Query 확인]
사례: case_004
영상: video_003
첫 후보 query: query_013
------------------------------
사례: case_005
영상: video_003
첫 후보 query: 없음
------------------------------
사례

In [27]:

from pathlib import Path
import json
import re

print("===== 24-2단계. 기존 Colab 검색어 탐색 =====")

NOTEBOOK = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/"
    "PinPoint_Korean_Video_Test.ipynb"
)

if not NOTEBOOK.is_file():
    raise FileNotFoundError(f"기존 노트북이 없습니다: {NOTEBOOK}")

with NOTEBOOK.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

patterns = [
    r"두부",
    r"\btofu\b",
    r"query_en",
    r"english_query",
    r"test_queries",
    r"test_cases",
    r"video_003",
]

matches = []

for index, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    text = "".join(source) if isinstance(source, list) else str(source)

    found = [
        pattern
        for pattern in patterns
        if re.search(pattern, text, flags=re.IGNORECASE)
    ]

    if found:
        matches.append((index, text, found))

print("관련 코드 셀:", len(matches), "개")

for index, text, found in matches[:35]:
    print("\n" + "=" * 65)
    print(f"노트북 셀 번호: {index + 1}")
    print("검색된 항목:", ", ".join(found))

    lines = text.splitlines()
    selected = set()

    for i, line in enumerate(lines):
        if any(
            re.search(p, line, flags=re.IGNORECASE)
            for p in patterns
        ):
            for j in range(max(0, i - 2), min(len(lines), i + 4)):
                selected.add(j)

    for j in sorted(selected)[:45]:
        print(f"{j + 1:3d}: {lines[j][:220]}")

if len(matches) > 35:
    print(f"\n나머지 관련 셀 {len(matches) - 35}개는 생략")

print("\n===== 24-2단계 완료 =====")

===== 24-2단계. 기존 Colab 검색어 탐색 =====
관련 코드 셀: 164 개

노트북 셀 번호: 11
검색된 항목: 두부, \btofu\b
  9:     name = unicodedata.normalize("NFC", path.name)
 10: 
 11:     if "두부" in name:
 12:         videos["tofu"] = path
 13:     elif "주방" in name:
 14:         videos["kitchen"] = path
 15:     elif "다이소" in name:

노트북 셀 번호: 15
검색된 항목: 두부, \btofu\b
  1: import cv2
  2: 
  3: video_path = str(videos["tofu"])
  4: cap = cv2.VideoCapture(video_path)
  5: 
  6: fps = cap.get(cv2.CAP_PROP_FPS)
  7: frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
  8: 
  9: print("영상:", "두부요리영상.mp4")
 10: print("FPS:", fps)
 11: print("영상 길이:", round(frames / fps, 2), "초")
 12: print("해상도:", int(cap.get(3)), "x", int(cap.get(4)))

노트북 셀 번호: 18
검색된 항목: \btofu\b
  2: 
  3: feature_path = build_feature(
  4:     video_path=str(videos["tofu"]),
  5:     video_id=video_id
  6: )
  7: 

노트북 셀 번호: 19
검색된 항목: \btofu\b
  7:     video_feature_path=feature_path,
  8:     subtitle_json_path=None,
  9:     query="cutting tofu with a knif

In [28]:

from pathlib import Path
import json
import traceback

print("===== 24-3단계. 두부 영상 Visual Top-3 실제 추론 =====")

FEATURE = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/clip_features/"
    "kr_tofu_001_clip_features.pt"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "04_PinPoint_ConsensusV3_MultilingualCLIP/"
    "02_results"
)

QUERY_KO = "두부를 칼로 자르는 장면"
QUERY_EN = "cutting tofu with a knife"

VIDEO_ID = "kr_tofu_001"
VIDEO_DURATION = 165.0

print("\n[1] 입력 확인")
print("영상:", VIDEO_ID)
print("한국어:", QUERY_KO)
print("영어:", QUERY_EN)
print("영상 길이:", VIDEO_DURATION)
print("CLIP 특징:", FEATURE.exists())

if not FEATURE.is_file():
    raise FileNotFoundError(FEATURE)

if inference is None:
    raise RuntimeError("Moment-DETR 모듈이 연결되지 않았습니다.")

print("\n[2] Moment-DETR Visual Top-3 실행")

try:
    visual_top3 = inference.predict_moments(
        video_id=VIDEO_ID,
        video_feature_path=str(FEATURE),
        query=QUERY_EN,
        video_duration=VIDEO_DURATION,
        top_k=3
    )

    print("✅ 추론 성공")
    print("후보 수:", len(visual_top3))

    for rank, candidate in enumerate(visual_top3, 1):
        print(
            f"{rank}위 | "
            f"{candidate['start_time']:.2f}~"
            f"{candidate['end_time']:.2f}초 | "
            f"score={candidate['score']:.4f}"
        )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    output_path = OUTPUT_DIR / "tofu_visual_top3_step24.json"

    if output_path.exists():
        raise FileExistsError(
            f"기존 결과가 존재하여 덮어쓰지 않습니다: {output_path}"
        )

    result_data = {
        "experiment": "Consensus V3 - Visual candidate extraction",
        "video_id": VIDEO_ID,
        "query_ko": QUERY_KO,
        "query_en": QUERY_EN,
        "video_duration": VIDEO_DURATION,
        "method": "Moment-DETR Visual-only",
        "checkpoint": str(inference.CHECKPOINT_PATH),
        "top_k": 3,
        "predictions": visual_top3
    }

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(
            result_data,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("\n✅ Google Drive 저장 완료")
    print(output_path)

except Exception as e:
    print("\n❌ 추론 또는 저장 실패")
    print("오류 종류:", type(e).__name__)
    print("오류 내용:", str(e))
    traceback.print_exc(limit=8)

print("\n===== 24-3단계 완료 =====")

===== 24-3단계. 두부 영상 Visual Top-3 실제 추론 =====

[1] 입력 확인
영상: kr_tofu_001
한국어: 두부를 칼로 자르는 장면
영어: cutting tofu with a knife
영상 길이: 165.0
CLIP 특징: True

[2] Moment-DETR Visual Top-3 실행
[AI] Loading CLIP on cuda
Loading CLIP models


100%|████████████████████████████████████████| 354M/354M [00:02<00:00, 147MiB/s]


✅ 추론 성공
후보 수: 3
1위 | 101.77~118.97초 | score=0.9842
2위 | 86.53~102.79초 | score=0.9608
3위 | 116.19~135.90초 | score=0.7395

✅ Google Drive 저장 완료
/content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/02_results/tofu_visual_top3_step24.json

===== 24-3단계 완료 =====


In [29]:

from pathlib import Path
import json
import traceback

print("===== 25단계. MiniLM 모델 및 두부 자막 준비 =====")

DRIVE = Path("/content/drive/MyDrive/pinpoint_capstone")

MODEL_DIR = (
    DRIVE / "03_Korean_Training_Data"
    / "minilm_finetuned/final_model"
)

SUBTITLE_PATH = (
    DRIVE / "02_Korean_Video_Test"
    / "kr_tofu_001_subtitles.json"
)

print("\n[1] 파일 존재 확인")
print("MiniLM 모델:", MODEL_DIR.is_dir())
print("두부 자막:", SUBTITLE_PATH.is_file())

if not MODEL_DIR.is_dir():
    raise FileNotFoundError(MODEL_DIR)

if not SUBTITLE_PATH.is_file():
    raise FileNotFoundError(SUBTITLE_PATH)

print("\n[2] 자막 JSON 구조 확인")

with SUBTITLE_PATH.open("r", encoding="utf-8") as f:
    subtitle_data = json.load(f)

print("최상위 자료형:", type(subtitle_data).__name__)

if isinstance(subtitle_data, list):
    print("자막 항목 수:", len(subtitle_data))
    print("첫 번째 항목:")
    print(json.dumps(
        subtitle_data[:2],
        ensure_ascii=False,
        indent=2
    )[:2000])

elif isinstance(subtitle_data, dict):
    print("최상위 키:", list(subtitle_data.keys()))
    for key, value in subtitle_data.items():
        if isinstance(value, list):
            print(f"{key}: {len(value)}개")
            print(json.dumps(
                value[:2],
                ensure_ascii=False,
                indent=2
            )[:2000])

print("\n[3] Fine-tuned MiniLM 모델 로딩")

try:
    from sentence_transformers import SentenceTransformer

    minilm_model = SentenceTransformer(str(MODEL_DIR))

    print("✅ Fine-tuned MiniLM 로딩 성공")
    print(
        "임베딩 차원:",
        minilm_model.get_sentence_embedding_dimension()
    )
    print("최대 입력 길이:", minilm_model.max_seq_length)

except Exception as e:
    minilm_model = None
    print("❌ MiniLM 로딩 실패")
    print("오류:", type(e).__name__, str(e))
    traceback.print_exc(limit=5)

print("\n===== 25단계 완료 =====")

===== 25단계. MiniLM 모델 및 두부 자막 준비 =====

[1] 파일 존재 확인
MiniLM 모델: True
두부 자막: True

[2] 자막 JSON 구조 확인
최상위 자료형: list
자막 항목 수: 57
첫 번째 항목:
[
  {
    "start_time": 0.0,
    "end_time": 4.74,
    "text": "여러분 잘 보세요 두부를 팔팔 끓는 물에 한번 삶아주기만 하면"
  },
  {
    "start_time": 4.74,
    "end_time": 8.7,
    "text": "순식간에 정말 맛있는 두부 요리 하나가 완성돼요"
  }
]

[3] Fine-tuned MiniLM 모델 로딩


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Fine-tuned MiniLM 로딩 성공
임베딩 차원: 384
최대 입력 길이: 128

===== 25단계 완료 =====


/tmp/ipykernel_640/4204796401.py:66: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  minilm_model.get_sentence_embedding_dimension()


In [30]:

from pathlib import Path
import json
import importlib
import traceback

print("===== 26단계. 두부 영상 MiniLM-KO·EN Top-3 =====")

REPO = Path("/content/pinpoint-capstone-ai")
ROOT = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test"
)
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "04_PinPoint_ConsensusV3_MultilingualCLIP/"
    "02_results"
)

QUERY_KO = "두부를 칼로 자르는 장면"
QUERY_EN = "cutting tofu with a knife"

SUBTITLE_PATH = ROOT / "kr_tofu_001_subtitles.json"
VISUAL_PATH = OUTPUT_DIR / "tofu_visual_top3_step24.json"
OUTPUT_PATH = OUTPUT_DIR / "tofu_minilm_top3_step26.json"

print("\n[1] 입력 파일 확인")

for name, path in [
    ("자막", SUBTITLE_PATH),
    ("Visual Top-3", VISUAL_PATH),
]:
    print("✅" if path.is_file() else "❌", name, path.name)

    if not path.is_file():
        raise FileNotFoundError(path)

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"기존 결과를 보호하기 위해 덮어쓰지 않습니다: {OUTPUT_PATH}"
    )

if minilm_model is None:
    raise RuntimeError(
        "25단계에서 Fine-tuned MiniLM을 불러오지 못했습니다."
    )

with SUBTITLE_PATH.open("r", encoding="utf-8") as f:
    subtitles = json.load(f)

with VISUAL_PATH.open("r", encoding="utf-8") as f:
    visual_data = json.load(f)

assert len(visual_data["predictions"]) == 3
assert visual_data["query_en"] == QUERY_EN

print("\n[2] GitHub Fusion 함수 연결")

fusion = importlib.import_module(
    "ai.features.fusion_search"
)

print("✅ Fusion 모듈:", fusion.__file__)

if Path(fusion.__file__).resolve() != (
    REPO / "ai/features/fusion_search.py"
).resolve():
    raise RuntimeError(
        "의도한 AI 브랜치의 Fusion 코드가 아닙니다."
    )

print("\n[3] MiniLM-KO Top-3 검색")

ko_top3 = fusion.get_semantic_subtitle_candidates(
    query=QUERY_KO,
    subtitles=subtitles,
    model=minilm_model,
    top_k=3
)

for rank, c in enumerate(ko_top3, 1):
    print(
        f"{rank}위 | {c['start_time']:.2f}~"
        f"{c['end_time']:.2f}초 | "
        f"score={c['score']:.4f}"
    )
    print("   자막:", c["text"][:120])

print("\n[4] MiniLM-EN Top-3 검색")

en_top3 = fusion.get_semantic_subtitle_candidates(
    query=QUERY_EN,
    subtitles=subtitles,
    model=minilm_model,
    top_k=3
)

for rank, c in enumerate(en_top3, 1):
    print(
        f"{rank}위 | {c['start_time']:.2f}~"
        f"{c['end_time']:.2f}초 | "
        f"score={c['score']:.4f}"
    )
    print("   자막:", c["text"][:120])

print("\n[5] 결과 저장")

result = {
    "video_id": "kr_tofu_001",
    "model_type": "finetuned",
    "query_ko": QUERY_KO,
    "query_en": QUERY_EN,
    "subtitle_count": len(subtitles),
    "visual_source_file": VISUAL_PATH.name,
    "ko_top3": ko_top3,
    "en_top3": en_top3
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("x", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("✅ 저장 완료:", OUTPUT_PATH)
print("\n===== 26단계 완료 =====")

===== 26단계. 두부 영상 MiniLM-KO·EN Top-3 =====

[1] 입력 파일 확인
✅ 자막 kr_tofu_001_subtitles.json
✅ Visual Top-3 tofu_visual_top3_step24.json

[2] GitHub Fusion 함수 연결
✅ Fusion 모듈: /content/pinpoint-capstone-ai/ai/features/fusion_search.py

[3] MiniLM-KO Top-3 검색
1위 | 4.74~8.70초 | score=0.3667
   자막: 순식간에 정말 맛있는 두부 요리 하나가 완성돼요
2위 | 23.72~26.46초 | score=0.3510
   자막: 제가 산두부는 한 모에 600번 정도 줬는데요
3위 | 21.06~26.46초 | score=0.3500
   자막: 다 썰었으면 접시에 담아 잠시 놔둘게요 제가 산두부는 한 모에 600번 정도 줬는데요

[4] MiniLM-EN Top-3 검색
1위 | 12.88~16.32초 | score=0.3233
   자막: 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요
2위 | 12.88~19.02초 | score=0.3229
   자막: 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요
3위 | 8.70~19.02초 | score=0.3225
   자막: 고소한 향의 식감까지 부드러워서 생선이나 고기 요리보다 더 맛있어요 먼저 두부 한 모를 준비해주세요 두부는 먼저 반으로 잘라주세요 그다음 다시 이렇게 넓적하고 큼직하게 썰어주세요

[5] 결과 저장
✅ 저장 완료: /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/02_results/tofu_minilm_top3_step26.json

===== 26단계 완료 =====


In [31]:

from pathlib import Path
import json
import importlib.util

print("===== 27단계. 실제 두부 영상 Consensus V3 =====")

BASE = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "04_PinPoint_ConsensusV3_MultilingualCLIP"
)

RESULT_DIR = BASE / "02_results"
EVAL_DIR = BASE / "03_evaluation"
CODE_PATH = BASE / "01_code/consensus_v3.py"

VISUAL_PATH = RESULT_DIR / "tofu_visual_top3_step24.json"
MINILM_PATH = RESULT_DIR / "tofu_minilm_top3_step26.json"

OUTPUT_PATH = EVAL_DIR / "tofu_consensus_v3_step27.json"

# 1. 입력 파일 검증
for path in [CODE_PATH, VISUAL_PATH, MINILM_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"기존 평가를 보호합니다: {OUTPUT_PATH}"
    )

with VISUAL_PATH.open("r", encoding="utf-8") as f:
    visual_data = json.load(f)

with MINILM_PATH.open("r", encoding="utf-8") as f:
    minilm_data = json.load(f)

assert visual_data["video_id"] == minilm_data["video_id"]
assert visual_data["query_ko"] == minilm_data["query_ko"]
assert visual_data["query_en"] == minilm_data["query_en"]

visual = visual_data["predictions"]
ko = minilm_data["ko_top3"]
en = minilm_data["en_top3"]

print("\n[1] 원본 후보 수")
print("Visual:", len(visual))
print("MiniLM-KO:", len(ko))
print("MiniLM-EN:", len(en))

# 2. 저장해 둔 V3 함수 불러오기
spec = importlib.util.spec_from_file_location(
    "pinpoint_consensus_v3",
    CODE_PATH
)

module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

# 중요:
# 이 단계에서는 정답 구간을 사용하지 않음
ranked = module.rank_by_consensus_v3(
    visual=visual,
    ko=ko,
    en=en,
    overlap_threshold=0.5
)

print("\n[2] Consensus V3 재순위 결과")

for rank, c in enumerate(ranked, 1):
    print(
        f"{rank}위 | "
        f"{c['rank_source']:6s} | "
        f"{c['start_time']:.2f}~"
        f"{c['end_time']:.2f}초 | "
        f"V3={c['consensus_score']:.4f} | "
        f"동의={c['support_sources']}"
    )

# 3. 예측이 완료된 이후에만 정답 사용
GT = {
    "start_time": 12.0,
    "end_time": 23.0
}

def temporal_iou(a, b):
    intersection = max(
        0.0,
        min(a["end_time"], b["end_time"])
        - max(a["start_time"], b["start_time"])
    )

    union = (
        a["end_time"] - a["start_time"]
        + b["end_time"] - b["start_time"]
        - intersection
    )

    return intersection / union if union > 0 else 0.0

evaluated = []

for rank, c in enumerate(ranked, 1):
    item = dict(c)
    item["final_rank"] = rank
    item["iou_with_ground_truth"] = temporal_iou(c, GT)
    evaluated.append(item)

top1_iou = evaluated[0]["iou_with_ground_truth"]
best5_iou = max(
    c["iou_with_ground_truth"] for c in evaluated[:5]
)

r1 = int(top1_iou >= 0.5)
r5 = int(best5_iou >= 0.5)

print("\n[3] 정답 구간 평가")
print("정답:", GT)
print("Top-1 IoU:", round(top1_iou, 4))
print("Top-5 최고 IoU:", round(best5_iou, 4))
print("R1@0.5:", r1)
print("R5@0.5:", r5)

print("\n[4] 결과 저장")

EVAL_DIR.mkdir(parents=True, exist_ok=True)

output = {
    "video_id": visual_data["video_id"],
    "query_ko": visual_data["query_ko"],
    "query_en": visual_data["query_en"],
    "method": "Consensus V3 - Visual KO EN Top3",
    "overlap_threshold": 0.5,
    "ground_truth": GT,
    "metrics": {
        "top1_iou": top1_iou,
        "best5_iou": best5_iou,
        "r1_at_05": r1,
        "r5_at_05": r5
    },
    "ranked_candidates": evaluated
}

with OUTPUT_PATH.open("x", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ 저장 완료:", OUTPUT_PATH)
print("\n===== 27단계 완료 =====")

===== 27단계. 실제 두부 영상 Consensus V3 =====

[1] 원본 후보 수
Visual: 3
MiniLM-KO: 3
MiniLM-EN: 3

[2] Consensus V3 재순위 결과
1위 | visual | 101.77~118.97초 | V3=1.0000 | 동의=[]
2위 | ko     | 4.74~8.70초 | V3=1.0000 | 동의=[]
3위 | en     | 12.88~16.32초 | V3=1.0000 | 동의=[]
4위 | visual | 86.53~102.79초 | V3=0.5000 | 동의=[]
5위 | ko     | 23.72~26.46초 | V3=0.5000 | 동의=[]
6위 | en     | 12.88~19.02초 | V3=0.5000 | 동의=[]
7위 | visual | 116.19~135.90초 | V3=0.3333 | 동의=[]
8위 | ko     | 21.06~26.46초 | V3=0.3333 | 동의=[]
9위 | en     | 8.70~19.02초 | V3=0.3333 | 동의=[]

[3] 정답 구간 평가
정답: {'start_time': 12.0, 'end_time': 23.0}
Top-1 IoU: 0.0
Top-5 최고 IoU: 0.3127
R1@0.5: 0
R5@0.5: 0

[4] 결과 저장
✅ 저장 완료: /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/03_evaluation/tofu_consensus_v3_step27.json

===== 27단계 완료 =====


# V1·V2·V3 성능 비교
- 기존 12개 검색어에 대해 Top-1 IoU, R1@0.5, Top-5 최고 IoU 비교

In [32]:

from pathlib import Path
import json
import importlib
import importlib.util

print("===== 28단계. 동일 후보 V1·V2·V3 비교 =====")

BASE = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "04_PinPoint_ConsensusV3_MultilingualCLIP"
)

RESULTS = BASE / "02_results"
EVAL = BASE / "03_evaluation"

VISUAL_FILE = RESULTS / "tofu_visual_top3_step24.json"
MINILM_FILE = RESULTS / "tofu_minilm_top3_step26.json"
V3_CODE = BASE / "01_code/consensus_v3.py"

OUTPUT = EVAL / "tofu_consensus_v1_v2_v3_step28.json"

if OUTPUT.exists():
    raise FileExistsError(
        f"기존 결과 보호: {OUTPUT}"
    )

with VISUAL_FILE.open("r", encoding="utf-8") as f:
    visual_data = json.load(f)

with MINILM_FILE.open("r", encoding="utf-8") as f:
    minilm_data = json.load(f)

assert visual_data["video_id"] == minilm_data["video_id"]
assert visual_data["query_en"] == minilm_data["query_en"]

visual = visual_data["predictions"]
ko = minilm_data["ko_top3"]
en = minilm_data["en_top3"]

# GitHub AI 브랜치의 기존 V1·V2
old = importlib.import_module(
    "ai.features.consensus_rerank"
)

# 새로 만든 V3
spec = importlib.util.spec_from_file_location(
    "consensus_v3_compare",
    V3_CODE
)
v3_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v3_module)

methods = {
    "V1_same_candidates":
        old.rank_by_consensus(visual, ko, en),

    "V2_same_candidates":
        old.rank_by_consensus_v2(visual, ko, en),

    "V3_same_candidates":
        v3_module.rank_by_consensus_v3(visual, ko, en)
}

GT = {"start_time": 12.0, "end_time": 23.0}

def iou(a, b):
    inter = max(
        0.0,
        min(float(a["end_time"]), float(b["end_time"]))
        - max(float(a["start_time"]), float(b["start_time"]))
    )
    union = (
        float(a["end_time"]) - float(a["start_time"])
        + float(b["end_time"]) - float(b["start_time"])
        - inter
    )
    return inter / union if union > 0 else 0.0

comparison = {}

for name, ranked in methods.items():
    top1 = iou(ranked[0], GT)
    best5 = max(iou(c, GT) for c in ranked[:5])
    best9 = max(iou(c, GT) for c in ranked)

    comparison[name] = {
        "top1_iou": top1,
        "best5_iou": best5,
        "best9_iou": best9,
        "r1_at_05": int(top1 >= 0.5),
        "r5_at_05": int(best5 >= 0.5),
        "ranked_candidates": [
            {
                "rank": i,
                "source": c["rank_source"],
                "start_time": c["start_time"],
                "end_time": c["end_time"],
                "consensus_score": c["consensus_score"],
                "iou": iou(c, GT)
            }
            for i, c in enumerate(ranked, 1)
        ]
    }

    print(f"\n[{name}]")
    print("Top-1 IoU:", round(top1, 4))
    print("Top-5 최고 IoU:", round(best5, 4))
    print("전체 9개 최고 IoU:", round(best9, 4))

    for c in comparison[name]["ranked_candidates"][:5]:
        print(
            f" {c['rank']}위 "
            f"{c['source']:6s} "
            f"{c['start_time']:.2f}~{c['end_time']:.2f}초 "
            f"IoU={c['iou']:.4f}"
        )

EVAL.mkdir(parents=True, exist_ok=True)

with OUTPUT.open("x", encoding="utf-8") as f:
    json.dump(
        {
            "video_id": visual_data["video_id"],
            "query_ko": visual_data["query_ko"],
            "query_en": visual_data["query_en"],
            "ground_truth": GT,
            "note": (
                "All methods receive identical Visual, KO, EN Top-3. "
                "This is not a reproduction of historical Hybrid-based evaluations."
            ),
            "comparison": comparison
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n✅ 비교 결과 저장:", OUTPUT)
print("===== 28단계 완료 =====")

===== 28단계. 동일 후보 V1·V2·V3 비교 =====

[V1_same_candidates]
Top-1 IoU: 0.0
Top-5 최고 IoU: 0.3127
전체 9개 최고 IoU: 0.5582
 1위 hybrid 101.77~118.97초 IoU=0.0000
 2위 ko     4.74~8.70초 IoU=0.0000
 3위 en     12.88~16.32초 IoU=0.3127
 4위 hybrid 86.53~102.79초 IoU=0.0000
 5위 ko     23.72~26.46초 IoU=0.0000

[V2_same_candidates]
Top-1 IoU: 0.0
Top-5 최고 IoU: 0.3127
전체 9개 최고 IoU: 0.5582
 1위 hybrid 101.77~118.97초 IoU=0.0000
 2위 ko     4.74~8.70초 IoU=0.0000
 3위 en     12.88~16.32초 IoU=0.3127
 4위 hybrid 86.53~102.79초 IoU=0.0000
 5위 ko     23.72~26.46초 IoU=0.0000

[V3_same_candidates]
Top-1 IoU: 0.0
Top-5 최고 IoU: 0.3127
전체 9개 최고 IoU: 0.5582
 1위 visual 101.77~118.97초 IoU=0.0000
 2위 ko     4.74~8.70초 IoU=0.0000
 3위 en     12.88~16.32초 IoU=0.3127
 4위 visual 86.53~102.79초 IoU=0.0000
 5위 ko     23.72~26.46초 IoU=0.0000

✅ 비교 결과 저장: /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/03_evaluation/tofu_consensus_v1_v2_v3_step28.json
===== 28단계 완료 =====


In [33]:

from pathlib import Path
import json
import re

print("===== 29단계. 기존 실제 검색어 및 영상 설정 확인 =====")

NOTEBOOK = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/"
    "PinPoint_Korean_Video_Test.ipynb"
)

with NOTEBOOK.open("r", encoding="utf-8") as f:
    nb = json.load(f)

matches = []

for index, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    text = "".join(source) if isinstance(source, list) else str(source)

    has_query_pair = (
        "korean_query=" in text
        and "english_query=" in text
    )

    has_case_definition = any(
        key in text for key in [
            "test_cases =",
            "cases =",
            "query_pairs =",
            "evaluation_cases =",
        ]
    )

    if has_query_pair or has_case_definition:
        matches.append((index + 1, text))

print("관련 코드 셀 수:", len(matches))

for cell_number, text in matches:
    print("\n" + "=" * 65)
    print("셀 번호:", cell_number)

    lines = text.splitlines()

    # 전체 노트북을 과도하게 출력하지 않도록
    # 앞부분만 표시하되 검색 조건을 확인할 수 있게 유지
    for line in lines[:35]:
        print(line[:220])

    if len(lines) > 35:
        print(f"... 나머지 {len(lines)-35}줄 생략")

print("\n===== 29단계 완료 =====")

===== 29단계. 기존 실제 검색어 및 영상 설정 확인 =====
관련 코드 셀 수: 64

셀 번호: 76
daiso_multilingual = get_multilingual_subtitle_candidates(
    korean_query="여행용 화장품 파우치를 보여주는 장면",
    english_query="a travel makeup pouch",
    subtitles=daiso_subtitles,
    model=semantic_model,
    top_k=3
)

for r in daiso_multilingual:
    print(
        f'{r["language"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'유사도={r["score"]:.4f}'
    )
    print("자막:", r["text"])
    print()

셀 번호: 78
tofu_multilingual = get_multilingual_subtitle_candidates(
    korean_query="두부를 칼로 자르는 장면",
    english_query="cutting tofu with a knife",
    subtitles=subtitles,
    model=semantic_model,
    top_k=3
)

tofu_unique = deduplicate_multilingual_candidates(
    tofu_multilingual
)

print("중복 제거 전:", len(tofu_multilingual))
print("중복 제거 후:", len(tofu_unique))

for r in tofu_unique:
    print(
        f'{r["language"]} | '
        f'{r["start_time"]:.2f}~{r["end_time"]:.2f}초 | '
        f'유사도={r["score"]:.

In [34]:

from pathlib import Path
import json

print("===== 30단계. 기존 6개 검색 사례 정리 =====")

DRIVE = Path("/content/drive/MyDrive/pinpoint_capstone")

OLD = DRIVE / "02_Korean_Video_Test"

BASE = (
    DRIVE /
    "04_PinPoint_ConsensusV3_MultilingualCLIP"
)

OUTPUT = BASE / "03_evaluation/consensus_v3_6cases_manifest.json"

if OUTPUT.exists():
    raise FileExistsError(
        f"기존 파일을 보호합니다: {OUTPUT}"
    )

# 기존 정답 JSON 불러오기
def read_json(filename):
    path = OLD / filename
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

baseline = read_json("fusion_baseline_evaluation.json")
seasoning = read_json("seasoning_evaluation.json")
rice = read_json("rice_washing_evaluation.json")
detergent = read_json("detergent_evaluation.json")

# 같은 영상에서 동일한 모델 설정으로 비교
video_info = {
    "kr_tofu_001": {
        "duration": 164.98,
        "feature": "kr_tofu_001_clip_features.pt",
        "subtitle": "kr_tofu_001_subtitles.json"
    },
    "kr_kitchen_001": {
        "duration": 37.08,
        "feature": "kr_kitchen_001_clip_features.pt",
        "subtitle": "kr_kitchen_001_subtitles.json"
    },
    "kr_daiso_001": {
        "duration": 32.08,
        "feature": "kr_daiso_001_clip_features.pt",
        "subtitle": "kr_daiso_001_subtitles.json"
    }
}

cases = [
    {
        "name": "두부 자르기",
        "video_id": "kr_tofu_001",
        "query_ko": "두부를 칼로 자르는 장면",
        "query_en": "cutting tofu with a knife",
        "ground_truth": baseline["ground_truth"]["두부"]
    },
    {
        "name": "두부 양념 준비",
        "video_id": "kr_tofu_001",
        "query_ko": seasoning["ground_truth"]["query_ko"],
        "query_en": seasoning["ground_truth"]["query_en"],
        "ground_truth": seasoning["ground_truth"]
    },
    {
        "name": "가위로 소스 짜기",
        "video_id": "kr_kitchen_001",
        "query_ko": "가위로 소스를 짜는 장면",
        "query_en": "squeezing sauce with scissors",
        "ground_truth": baseline["ground_truth"]["주방"]
    },
    {
        "name": "쌀 씻기",
        "video_id": "kr_kitchen_001",
        "query_ko": rice["ground_truth"]["query_ko"],
        "query_en": rice["ground_truth"]["query_en"],
        "ground_truth": rice["ground_truth"]
    },
    {
        "name": "다이소 파우치",
        "video_id": "kr_daiso_001",
        "query_ko": "여행용 화장품 파우치를 보여주는 장면",
        "query_en": "a travel makeup pouch",
        "ground_truth": baseline["ground_truth"]["다이소"]
    },
    {
        "name": "다이소 시트 세제",
        "video_id": "kr_daiso_001",
        "query_ko": detergent["ground_truth"]["query_ko"],
        "query_en": detergent["ground_truth"]["query_en"],
        "ground_truth": detergent["ground_truth"]
    }
]

print("\n[1] 검색 사례 및 정답 확인")

for case in cases:
    video_id = case["video_id"]
    info = video_info[video_id]

    feature_path = (
        OLD / "clip_features" / info["feature"]
    )
    subtitle_path = OLD / info["subtitle"]

    gt = case["ground_truth"]

    assert feature_path.is_file(), feature_path
    assert subtitle_path.is_file(), subtitle_path
    assert case["query_ko"].strip()
    assert case["query_en"].strip()
    assert 0 <= float(gt["start_time"]) < float(gt["end_time"])
    assert float(gt["end_time"]) <= info["duration"] + 1.0

    print(f"\n검색 장면: {case['name']}")
    print("영상:", video_id)
    print("한국어:", case["query_ko"])
    print("영어:", case["query_en"])
    print(
        "정답:",
        gt["start_time"], "~",
        gt["end_time"], "초"
    )

print("\n[2] 평가 목록 저장")

output = {
    "experiment": "Consensus V1 V2 V3 comparison",
    "case_count": len(cases),
    "candidate_sources": [
        "Moment-DETR visual",
        "Fine-tuned MiniLM KO",
        "Fine-tuned MiniLM EN"
    ],
    "video_info": video_info,
    "cases": cases,
    "evaluation_note": (
        "Previously used development cases. "
        "Ground truth is used only after ranking."
    )
}

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT.open("x", encoding="utf-8") as f:
    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ 저장 완료:", OUTPUT)
print("총 검색 사례:", len(cases))
print("\n===== 30단계 완료 =====")

===== 30단계. 기존 6개 검색 사례 정리 =====

[1] 검색 사례 및 정답 확인

검색 장면: 두부 자르기
영상: kr_tofu_001
한국어: 두부를 칼로 자르는 장면
영어: cutting tofu with a knife
정답: 12.0 ~ 23.0 초

검색 장면: 두부 양념 준비
영상: kr_tofu_001
한국어: 두부 양념을 준비하는 장면
영어: preparing seasoning sauce for tofu
정답: 95.0 ~ 135.0 초

검색 장면: 가위로 소스 짜기
영상: kr_kitchen_001
한국어: 가위로 소스를 짜는 장면
영어: squeezing sauce with scissors
정답: 5.96 ~ 11.32 초

검색 장면: 쌀 씻기
영상: kr_kitchen_001
한국어: 쌀을 씻는 장면
영어: washing rice
정답: 29.0 ~ 32.0 초

검색 장면: 다이소 파우치
영상: kr_daiso_001
한국어: 여행용 화장품 파우치를 보여주는 장면
영어: a travel makeup pouch
정답: 3.08 ~ 8.4 초

검색 장면: 다이소 시트 세제
영상: kr_daiso_001
한국어: 여행용 시트 세제를 보여주는 장면
영어: showing travel laundry detergent sheets
정답: 13.0 ~ 18.0 초

[2] 평가 목록 저장
✅ 저장 완료: /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/03_evaluation/consensus_v3_6cases_manifest.json
총 검색 사례: 6

===== 30단계 완료 =====


In [35]:

from pathlib import Path
import json
import re
import traceback

print("===== 31단계. 6개 검색어 Top-3 후보 생성 =====")

DRIVE = Path("/content/drive/MyDrive/pinpoint_capstone")

BASE = (
    DRIVE /
    "04_PinPoint_ConsensusV3_MultilingualCLIP"
)

MANIFEST = (
    BASE / "03_evaluation/"
    "consensus_v3_6cases_manifest.json"
)

DATA_DIR = DRIVE / "02_Korean_Video_Test"

OUTPUT_DIR = BASE / "02_results/consensus_v3_6cases"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 이전 단계에서 모델이 실제로 준비됐는지 확인
assert inference is not None, "Moment-DETR이 연결되지 않았습니다."
assert minilm_model is not None, "MiniLM이 연결되지 않았습니다."

with MANIFEST.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

cases = manifest["cases"]
video_info = manifest["video_info"]

assert len(cases) == 6

print("평가 검색어 수:", len(cases))

success = 0
skipped = 0
failed = 0

for case_index, case in enumerate(cases, 1):
    name = case["name"]
    video_id = case["video_id"]

    info = video_info[video_id]

    feature_path = (
        DATA_DIR / "clip_features" / info["feature"]
    )

    subtitle_path = DATA_DIR / info["subtitle"]

    safe_name = re.sub(
        r"[^a-zA-Z0-9_-]",
        "_",
        f"case_{case_index:02d}_{video_id}"
    )

    output_path = OUTPUT_DIR / f"{safe_name}.json"

    print("\n" + "=" * 65)
    print(f"[{case_index}/6] {name}")

    if output_path.exists():
        print("⏭️ 이미 결과 파일이 있어 건너뜁니다.")
        skipped += 1
        continue

    try:
        if not feature_path.is_file():
            raise FileNotFoundError(feature_path)

        if not subtitle_path.is_file():
            raise FileNotFoundError(subtitle_path)

        with subtitle_path.open("r", encoding="utf-8") as f:
            subtitles = json.load(f)

        query_ko = case["query_ko"]
        query_en = case["query_en"]

        # 1. Moment-DETR Visual Top-3
        visual = inference.predict_moments(
            video_id=video_id,
            video_feature_path=str(feature_path),
            query=query_en,
            video_duration=info["duration"],
            top_k=3
        )

        # 2. Fine-tuned MiniLM 한국어 Top-3
        ko = fusion.get_semantic_subtitle_candidates(
            query=query_ko,
            subtitles=subtitles,
            model=minilm_model,
            top_k=3
        )

        # 3. Fine-tuned MiniLM 영어 Top-3
        en = fusion.get_semantic_subtitle_candidates(
            query=query_en,
            subtitles=subtitles,
            model=minilm_model,
            top_k=3
        )

        # 입력 데이터 검증
        assert len(visual) == 3
        assert len(ko) == 3
        assert len(en) == 3

        for group in [visual, ko, en]:
            for candidate in group:
                start = float(candidate["start_time"])
                end = float(candidate["end_time"])

                assert 0 <= start < end
                assert end <= info["duration"] + 0.1

        output = {
            "name": name,
            "video_id": video_id,
            "query_ko": query_ko,
            "query_en": query_en,
            "video_duration": info["duration"],
            "visual_model": "Moment-DETR",
            "subtitle_model": "Fine-tuned MiniLM",
            "candidate_top_k": 3,
            "visual": visual,
            "ko": ko,
            "en": en
        }

        # 덮어쓰기 방지
        with output_path.open("x", encoding="utf-8") as f:
            json.dump(
                output,
                f,
                ensure_ascii=False,
                indent=2
            )

        print("✅ Visual:", len(visual))
        print("✅ MiniLM-KO:", len(ko))
        print("✅ MiniLM-EN:", len(en))

        print(
            "Visual 1위:",
            visual[0]["start_time"],
            "~",
            visual[0]["end_time"]
        )

        print(
            "KO 1위:",
            ko[0]["start_time"],
            "~",
            ko[0]["end_time"]
        )

        print(
            "EN 1위:",
            en[0]["start_time"],
            "~",
            en[0]["end_time"]
        )

        print("저장:", output_path.name)
        success += 1

    except Exception as e:
        failed += 1
        print("❌ 실패:", type(e).__name__, str(e))
        traceback.print_exc(limit=3)

print("\n===== 31단계 요약 =====")
print("새로 성공:", success)
print("기존 결과 건너뜀:", skipped)
print("실패:", failed)

print("총 파일 수:", len(list(OUTPUT_DIR.glob('*.json'))))

print("\n===== 31단계 완료 =====")

===== 31단계. 6개 검색어 Top-3 후보 생성 =====
평가 검색어 수: 6

[1/6] 두부 자르기
✅ Visual: 3
✅ MiniLM-KO: 3
✅ MiniLM-EN: 3
Visual 1위: 101.755 ~ 118.9581
KO 1위: 4.74 ~ 8.7
EN 1위: 12.88 ~ 16.32
저장: case_01_kr_tofu_001.json

[2/6] 두부 양념 준비
✅ Visual: 3
✅ MiniLM-KO: 3
✅ MiniLM-EN: 3
Visual 1위: 101.7545 ~ 119.1551
KO 1위: 137.58 ~ 143.58
EN 1위: 111.26 ~ 116.02
저장: case_02_kr_tofu_001.json

[3/6] 가위로 소스 짜기
✅ Visual: 3
✅ MiniLM-KO: 3
✅ MiniLM-EN: 3
Visual 1위: 4.4948 ~ 10.2665
KO 1위: 5.96 ~ 8.08
EN 1위: 23.32 ~ 26.56
저장: case_03_kr_kitchen_001.json

[4/6] 쌀 씻기
❌ 실패: AssertionError 

[5/6] 다이소 파우치


Traceback (most recent call last):
  File "/tmp/ipykernel_640/2666985840.py", line 111, in <cell line: 0>
    assert len(visual) == 3
           ^^^^^^^^^^^^^^^^
AssertionError


✅ Visual: 3
✅ MiniLM-KO: 3
✅ MiniLM-EN: 3
Visual 1위: 18.0719 ~ 24.5001
KO 1위: 2.08 ~ 7.28
EN 1위: 3.08 ~ 8.4
저장: case_05_kr_daiso_001.json

[6/6] 다이소 시트 세제
✅ Visual: 3
✅ MiniLM-KO: 3
✅ MiniLM-EN: 3
Visual 1위: 16.4597 ~ 24.5203
KO 1위: 13.72 ~ 15.16
EN 1위: 22.68 ~ 26.8
저장: case_06_kr_daiso_001.json

===== 31단계 요약 =====
새로 성공: 5
기존 결과 건너뜀: 0
실패: 1
총 파일 수: 5

===== 31단계 완료 =====


In [36]:

from pathlib import Path
import json
import traceback

print("===== 31-1단계. 쌀 씻기 후보 복구 =====")

DRIVE = Path("/content/drive/MyDrive/pinpoint_capstone")
BASE = DRIVE / "04_PinPoint_ConsensusV3_MultilingualCLIP"
DATA_DIR = DRIVE / "02_Korean_Video_Test"

MANIFEST = (
    BASE / "03_evaluation/consensus_v3_6cases_manifest.json"
)
OUTPUT_DIR = BASE / "02_results/consensus_v3_6cases"
OUTPUT_PATH = OUTPUT_DIR / "case_04_kr_kitchen_001.json"

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"기존 파일을 덮어쓰지 않습니다: {OUTPUT_PATH}"
    )

with MANIFEST.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

case = manifest["cases"][3]

assert case["name"] == "쌀 씻기"
assert case["video_id"] == "kr_kitchen_001"

info = manifest["video_info"][case["video_id"]]

feature_path = (
    DATA_DIR / "clip_features" / info["feature"]
)
subtitle_path = DATA_DIR / info["subtitle"]

print("\n[1] 검색 정보")
print("검색어:", case["query_ko"])
print("영어:", case["query_en"])
print("영상 길이:", info["duration"])
print("특징 파일:", feature_path.is_file())
print("자막 파일:", subtitle_path.is_file())

assert feature_path.is_file()
assert subtitle_path.is_file()

with subtitle_path.open("r", encoding="utf-8") as f:
    subtitles = json.load(f)

print("\n[2] 검색 방식별 후보 재생성")

visual = inference.predict_moments(
    video_id=case["video_id"],
    video_feature_path=str(feature_path),
    query=case["query_en"],
    video_duration=info["duration"],
    top_k=3
)

ko = fusion.get_semantic_subtitle_candidates(
    query=case["query_ko"],
    subtitles=subtitles,
    model=minilm_model,
    top_k=3
)

en = fusion.get_semantic_subtitle_candidates(
    query=case["query_en"],
    subtitles=subtitles,
    model=minilm_model,
    top_k=3
)

print("\n[3] 실제 반환 후보 수")
print("Visual:", len(visual))
print("MiniLM-KO:", len(ko))
print("MiniLM-EN:", len(en))

for name, candidates in [
    ("Visual", visual),
    ("MiniLM-KO", ko),
    ("MiniLM-EN", en)
]:
    print(f"\n[{name}]")
    for rank, c in enumerate(candidates, 1):
        print(
            f"{rank}위 | "
            f"{float(c['start_time']):.2f}~"
            f"{float(c['end_time']):.2f}초 | "
            f"score={float(c['score']):.4f}"
        )

print("\n[4] 유효성 검사")

for name, group in [
    ("Visual", visual),
    ("MiniLM-KO", ko),
    ("MiniLM-EN", en)
]:
    assert 1 <= len(group) <= 3, (
        f"{name} 후보 개수 이상: {len(group)}"
    )

    for c in group:
        start = float(c["start_time"])
        end = float(c["end_time"])
        assert 0 <= start < end
        assert end <= info["duration"] + 0.1

print("✅ 실제 반환 후보 모두 유효")

print("\n[5] Google Drive 저장")

output = {
    "name": case["name"],
    "video_id": case["video_id"],
    "query_ko": case["query_ko"],
    "query_en": case["query_en"],
    "video_duration": info["duration"],
    "visual_model": "Moment-DETR",
    "subtitle_model": "Fine-tuned MiniLM",
    "candidate_top_k": 3,
    "actual_candidate_counts": {
        "visual": len(visual),
        "ko": len(ko),
        "en": len(en)
    },
    "visual": visual,
    "ko": ko,
    "en": en
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("x", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ 저장 완료:", OUTPUT_PATH)
print("\n===== 31-1단계 완료 =====")

===== 31-1단계. 쌀 씻기 후보 복구 =====

[1] 검색 정보
검색어: 쌀을 씻는 장면
영어: washing rice
영상 길이: 37.08
특징 파일: True
자막 파일: True

[2] 검색 방식별 후보 재생성

[3] 실제 반환 후보 수
Visual: 2
MiniLM-KO: 3
MiniLM-EN: 3

[Visual]
1위 | 20.89~36.43초 | score=0.9413
2위 | 7.99~14.44초 | score=0.0024

[MiniLM-KO]
1위 | 29.28~31.72초 | score=0.7528
2위 | 29.28~33.52초 | score=0.6392
3위 | 26.56~31.72초 | score=0.6289

[MiniLM-EN]
1위 | 29.28~31.72초 | score=0.7476
2위 | 29.28~33.52초 | score=0.6461
3위 | 29.28~34.80초 | score=0.6287

[4] 유효성 검사
✅ 실제 반환 후보 모두 유효

[5] Google Drive 저장
✅ 저장 완료: /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/02_results/consensus_v3_6cases/case_04_kr_kitchen_001.json

===== 31-1단계 완료 =====


In [37]:

from pathlib import Path
import json
import importlib
import importlib.util
import numpy as np

print("===== 32단계. Consensus V1·V2·V3 전체 평가 =====")

BASE = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "04_PinPoint_ConsensusV3_MultilingualCLIP"
)

MANIFEST_PATH = (
    BASE / "03_evaluation/consensus_v3_6cases_manifest.json"
)
CANDIDATE_DIR = BASE / "02_results/consensus_v3_6cases"
V3_CODE = BASE / "01_code/consensus_v3.py"

OUTPUT = (
    BASE / "03_evaluation/"
    "consensus_v1_v2_v3_6cases_step32.json"
)

if OUTPUT.exists():
    raise FileExistsError(
        f"기존 결과를 덮어쓰지 않습니다: {OUTPUT}"
    )

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

old = importlib.import_module(
    "ai.features.consensus_rerank"
)

spec = importlib.util.spec_from_file_location(
    "consensus_v3_batch",
    V3_CODE
)
v3_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v3_module)

def temporal_iou(pred, gt):
    ps = float(pred["start_time"])
    pe = float(pred["end_time"])
    gs = float(gt["start_time"])
    ge = float(gt["end_time"])

    intersection = max(
        0.0, min(pe, ge) - max(ps, gs)
    )
    union = (pe - ps) + (ge - gs) - intersection

    return intersection / union if union > 0 else 0.0

methods = {
    "V1": old.rank_by_consensus,
    "V2": old.rank_by_consensus_v2,
    "V3": v3_module.rank_by_consensus_v3
}

# 모든 후보 파일을 먼저 확인
case_data = []

for i, case in enumerate(manifest["cases"], 1):
    video_id = case["video_id"]
    path = (
        CANDIDATE_DIR /
        f"case_{i:02d}_{video_id}.json"
    )

    if not path.is_file():
        raise FileNotFoundError(path)

    with path.open("r", encoding="utf-8") as f:
        candidate_data = json.load(f)

    assert candidate_data["name"] == case["name"]
    assert candidate_data["video_id"] == video_id
    assert candidate_data["query_ko"] == case["query_ko"]
    assert candidate_data["query_en"] == case["query_en"]

    case_data.append((case, candidate_data))

print("\n[1] 입력 검증 완료")
print("검색어 수:", len(case_data))
print("평가 방식:", ", ".join(methods))

results = {name: [] for name in methods}

print("\n[2] 검색어별 성능 비교")

for case, data in case_data:
    visual = data["visual"]
    ko = data["ko"]
    en = data["en"]
    gt = case["ground_truth"]

    print("\n" + "=" * 62)
    print("검색어:", case["name"])
    print(
        f"후보 수: Visual {len(visual)} / "
        f"KO {len(ko)} / EN {len(en)}"
    )

    for method_name, rank_function in methods.items():
        ranked = rank_function(visual, ko, en)

        if not ranked:
            raise RuntimeError(
                f"{case['name']}에서 {method_name} 결과가 비어 있습니다."
            )

        ious = [
            temporal_iou(c, gt)
            for c in ranked
        ]

        top1_iou = ious[0]
        best5_iou = max(ious[:5])
        best_all_iou = max(ious)

        row = {
            "name": case["name"],
            "video_id": case["video_id"],
            "candidate_count": len(ranked),
            "top1_iou": top1_iou,
            "best5_iou": best5_iou,
            "best_all_iou": best_all_iou,
            "r1_at_05": int(top1_iou >= 0.5),
            "r5_at_05": int(best5_iou >= 0.5),
            "oracle_at_05": int(best_all_iou >= 0.5),
            "top1_source": ranked[0]["rank_source"],
            "top1_start": ranked[0]["start_time"],
            "top1_end": ranked[0]["end_time"],
            "ranked_candidates": [
                {
                    **dict(candidate),
                    "final_rank": rank,
                    "iou_with_gt": iou
                }
                for rank, (candidate, iou)
                in enumerate(zip(ranked, ious), 1)
            ]
        }

        results[method_name].append(row)

        print(
            f"{method_name} | "
            f"Top1 IoU={top1_iou:.4f} | "
            f"Top5={best5_iou:.4f} | "
            f"Oracle={best_all_iou:.4f} | "
            f"1위={row['top1_source']}"
        )

print("\n[3] 전체 평균 성능")

summary = {}

for method_name, rows in results.items():
    n = len(rows)

    summary[method_name] = {
        "case_count": n,
        "mean_top1_iou": float(np.mean(
            [r["top1_iou"] for r in rows]
        )),
        "mean_best5_iou": float(np.mean(
            [r["best5_iou"] for r in rows]
        )),
        "r1_at_05": float(np.mean(
            [r["r1_at_05"] for r in rows]
        )),
        "r5_at_05": float(np.mean(
            [r["r5_at_05"] for r in rows]
        )),
        "oracle_at_05": float(np.mean(
            [r["oracle_at_05"] for r in rows]
        ))
    }

    s = summary[method_name]

    print(f"\n[{method_name}]")
    print("평균 Top-1 IoU:", round(s["mean_top1_iou"], 4))
    print("평균 Top-5 최고 IoU:", round(s["mean_best5_iou"], 4))
    print("R1@0.5:", f"{s['r1_at_05']*100:.1f}%")
    print("R5@0.5:", f"{s['r5_at_05']*100:.1f}%")
    print("Oracle@0.5:", f"{s['oracle_at_05']*100:.1f}%")

print("\n[4] 평가 결과 저장")

output_data = {
    "experiment": "Same candidate comparison - V1 V2 V3",
    "candidate_sources": [
        "Visual Moment-DETR",
        "Fine-tuned MiniLM KO",
        "Fine-tuned MiniLM EN"
    ],
    "case_count": len(case_data),
    "note": (
        "Same per-case candidates for all methods. "
        "Previously used development examples, not held-out test set. "
        "V1/V2 visual candidates may be labeled hybrid internally, "
        "but are pure Visual Moment-DETR predictions."
    ),
    "summary": summary,
    "results": results
}

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT.open("x", encoding="utf-8") as f:
    json.dump(
        output_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ 저장 완료:", OUTPUT)
print("\n===== 32단계 완료 =====")

===== 32단계. Consensus V1·V2·V3 전체 평가 =====

[1] 입력 검증 완료
검색어 수: 6
평가 방식: V1, V2, V3

[2] 검색어별 성능 비교

검색어: 두부 자르기
후보 수: Visual 3 / KO 3 / EN 3
V1 | Top1 IoU=0.0000 | Top5=0.3127 | Oracle=0.5582 | 1위=hybrid
V2 | Top1 IoU=0.0000 | Top5=0.3127 | Oracle=0.5582 | 1위=hybrid
V3 | Top1 IoU=0.0000 | Top5=0.3127 | Oracle=0.5582 | 1위=visual

검색어: 두부 양념 준비
후보 수: Visual 3 / KO 3 / EN 3
V1 | Top1 IoU=0.4350 | Top5=0.6546 | Oracle=0.6546 | 1위=hybrid
V2 | Top1 IoU=0.4350 | Top5=0.6546 | Oracle=0.6546 | 1위=hybrid
V3 | Top1 IoU=0.4350 | Top5=0.6546 | Oracle=0.6546 | 1위=visual

검색어: 가위로 소스 짜기
후보 수: Visual 3 / KO 3 / EN 3
V1 | Top1 IoU=0.6310 | Top5=0.6310 | Oracle=0.6310 | 1위=hybrid
V2 | Top1 IoU=0.6310 | Top5=0.6310 | Oracle=0.6310 | 1위=hybrid
V3 | Top1 IoU=0.6310 | Top5=0.6310 | Oracle=0.6310 | 1위=visual

검색어: 쌀 씻기
후보 수: Visual 2 / KO 3 / EN 3
V1 | Top1 IoU=0.8133 | Top5=0.8133 | Oracle=0.8133 | 1위=ko
V2 | Top1 IoU=0.8133 | Top5=0.8133 | Oracle=0.8133 | 1위=ko
V3 | Top1 IoU=0.8133 | Top5=0.8133 | Oracle=

In [38]:

from pathlib import Path
import json

print("===== 33단계. Consensus V3 Top-1 실패 원인 분석 =====")

BASE = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "04_PinPoint_ConsensusV3_MultilingualCLIP"
)

INPUT = (
    BASE / "03_evaluation/"
    "consensus_v1_v2_v3_6cases_step32.json"
)

OUTPUT = (
    BASE / "03_evaluation/"
    "consensus_v3_failure_diagnosis_step33.json"
)

if not INPUT.is_file():
    raise FileNotFoundError(INPUT)

if OUTPUT.exists():
    raise FileExistsError(
        f"기존 분석 파일 보호: {OUTPUT}"
    )

with INPUT.open("r", encoding="utf-8") as f:
    data = json.load(f)

rows = data["results"]["V3"]

def source_name(candidate):
    src = candidate.get("rank_source", "")
    return "visual" if src == "hybrid" else src

diagnostics = []

for row in rows:
    candidates = row["ranked_candidates"]
    top1_ok = row["r1_at_05"] == 1

    if top1_ok:
        continue

    print("\n" + "=" * 68)
    print("검색어:", row["name"])
    print("현재 V3 Top-1 IoU:", round(row["top1_iou"], 4))

    best = max(
        candidates,
        key=lambda c: c["iou_with_gt"]
    )

    print("\n[1] 실제 가장 정확했던 후보")
    print("V3 최종 순위:", best["final_rank"])
    print("검색 방식:", source_name(best))
    print(
        "시간:",
        round(best["start_time"], 2),
        "~",
        round(best["end_time"], 2),
        "초"
    )
    print("정답 IoU:", round(best["iou_with_gt"], 4))

    print("\n[2] 검색 방식별 가장 좋은 후보")

    source_best = {}

    for source in ("visual", "ko", "en"):
        group = [
            c for c in candidates
            if source_name(c) == source
        ]

        if not group:
            continue

        winner = max(
            group,
            key=lambda c: c["iou_with_gt"]
        )

        source_best[source] = {
            "final_rank": winner["final_rank"],
            "iou": winner["iou_with_gt"],
            "start_time": winner["start_time"],
            "end_time": winner["end_time"]
        }

        print(
            f"{source:6s} | "
            f"V3 {winner['final_rank']}위 | "
            f"IoU={winner['iou_with_gt']:.4f}"
        )

    print("\n[3] V3 상위 후보 순위 상세")

    for c in candidates[:9]:
        print(
            f"{c['final_rank']}위 | "
            f"{source_name(c):6s} | "
            f"{c['start_time']:.2f}~"
            f"{c['end_time']:.2f}초 | "
            f"V3={c['consensus_score']:.4f} | "
            f"GT IoU={c['iou_with_gt']:.4f} | "
            f"동의={c.get('support_sources', [])}"
        )

    diagnostics.append({
        "case_name": row["name"],
        "top1_iou": row["top1_iou"],
        "best_candidate": {
            "final_rank": best["final_rank"],
            "source": source_name(best),
            "start_time": best["start_time"],
            "end_time": best["end_time"],
            "iou": best["iou_with_gt"]
        },
        "best_by_source": source_best,
        "ranked_candidates": candidates
    })

print("\n" + "=" * 68)
print("[4] 실패 사례 요약")
print("Top-1 실패:", len(diagnostics), "/", len(rows))

for item in diagnostics:
    best = item["best_candidate"]
    print(
        f"{item['case_name']}: "
        f"정답에 가장 가까운 후보는 "
        f"{best['source']} / "
        f"V3 {best['final_rank']}위 / "
        f"IoU={best['iou']:.4f}"
    )

with OUTPUT.open("x", encoding="utf-8") as f:
    json.dump(
        {
            "experiment": "Consensus V3 failure diagnosis",
            "evaluation_cases": len(rows),
            "failed_top1_cases": len(diagnostics),
            "note": (
                "Ground truth is used for post-hoc diagnosis only, "
                "not candidate ranking."
            ),
            "diagnostics": diagnostics
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n✅ 분석 결과 저장:", OUTPUT)
print("===== 33단계 완료 =====")

===== 33단계. Consensus V3 Top-1 실패 원인 분석 =====

검색어: 두부 자르기
현재 V3 Top-1 IoU: 0.0

[1] 실제 가장 정확했던 후보
V3 최종 순위: 6
검색 방식: en
시간: 12.88 ~ 19.02 초
정답 IoU: 0.5582

[2] 검색 방식별 가장 좋은 후보
visual | V3 1위 | IoU=0.0000
ko     | V3 8위 | IoU=0.1342
en     | V3 6위 | IoU=0.5582

[3] V3 상위 후보 순위 상세
1위 | visual | 101.75~118.96초 | V3=1.0000 | GT IoU=0.0000 | 동의=[]
2위 | ko     | 4.74~8.70초 | V3=1.0000 | GT IoU=0.0000 | 동의=[]
3위 | en     | 12.88~16.32초 | V3=1.0000 | GT IoU=0.3127 | 동의=[]
4위 | visual | 86.52~102.78초 | V3=0.5000 | GT IoU=0.0000 | 동의=[]
5위 | ko     | 23.72~26.46초 | V3=0.5000 | GT IoU=0.0000 | 동의=[]
6위 | en     | 12.88~19.02초 | V3=0.5000 | GT IoU=0.5582 | 동의=[]
7위 | visual | 116.18~135.89초 | V3=0.3333 | GT IoU=0.0000 | 동의=[]
8위 | ko     | 21.06~26.46초 | V3=0.3333 | GT IoU=0.1342 | 동의=[]
9위 | en     | 8.70~19.02초 | V3=0.3333 | GT IoU=0.4909 | 동의=[]

검색어: 두부 양념 준비
현재 V3 Top-1 IoU: 0.435

[1] 실제 가장 정확했던 후보
V3 최종 순위: 4
검색 방식: visual
시간: 90.21 ~ 151.32 초
정답 IoU: 0.6546

[2] 검색 방식별 가장 좋은 후보
visual | V

# 기억할 사항
# PinPoint AI — Consensus V3 실험 결과 및 향후 개선 계획

## 1. 실험 개요

### 1.1 실험 목적

PinPoint AI는 사용자가 자연어로 원하는 장면을 검색하면 영상 내부에서 해당 장면의 시간 구간을 찾아 반환하는 시스템이다.

현재 AI 검색은 다음 세 가지 경로를 활용한다.

1. **Visual Moment-DETR:** 영상의 CLIP 특징과 영어 검색어를 이용해 관련 구간 탐색
2. **Fine-tuned MiniLM-KO:** 한국어 검색어와 한국어 자막의 의미적 유사도를 이용한 구간 탐색
3. **Fine-tuned MiniLM-EN:** 영어 검색어와 동일한 한국어 자막의 의미적 유사도를 이용한 구간 탐색

MiniLM-KO와 MiniLM-EN은 서로 다른 모델이 아니라, 동일한 Fine-tuned MiniLM에 서로 다른 언어의 검색어를 입력하는 두 검색 경로이다.

기존 Fusion은 서로 다른 검색 경로에서 생성한 후보를 번갈아 배치하고, 겹치는 후보를 제거하는 방식이어서 정확한 후보가 최종 결과에서 누락될 수 있었다.

이를 개선하기 위해 후보 간 시간적 일치 정도를 이용하는 Consensus V1, V2, V3를 실험하였다.

**Consensus V3의 핵심 목표는 세 검색 경로에서 생성한 후보를 모두 보존하고, 서로 다른 경로가 비슷한 시간대를 지목할 경우 가산점을 부여해 최종 Top-1 정확도를 개선하는 것이었다.**

---

## 2. Consensus V3 설계

### 2.1 입력 구조

각 검색어에 대해 다음 후보를 생성한다.

| 검색 경로 | 사용 모델 | 입력 | 후보 수 |
|---|---|---|---|
| Visual | Moment-DETR | 영어 검색어 + 영상 CLIP 특징 | 최대 Top-3 |
| KO | Fine-tuned MiniLM | 한국어 검색어 + 한국어 자막 | Top-3 |
| EN | Fine-tuned MiniLM | 영어 검색어 + 한국어 자막 | Top-3 |

총 최대 9개의 후보를 생성한다.

단, Moment-DETR의 NMS 처리로 인해 후보가 3개 미만 반환될 수 있다. 실제 쌀 씻기 검색에서는 Visual 후보가 2개만 생성되었다.

### 2.2 V3 재순위화 방식

Consensus V3는 다음 정보를 활용한다.

- **기본 순위 점수:** 검색 경로 내 원래 순위에 따라 `1/rank` 적용
- **검색 경로 간 시간적 겹침:** 서로 다른 경로 후보들의 Temporal IoU 계산
- **동의 가산점:** Temporal IoU가 0.5 이상일 때 적용
- **Visual과 자막 검색 간 동의:** 상대적으로 높은 가산점 부여
- **한국어·영어 자막 검색 간 동의:** 동일 MiniLM 모델을 사용한다는 점을 고려해 상대적으로 낮은 가산점 부여

현재 V3 구현의 주요 설정은 다음과 같다.

| 설정 | 값 |
|---|---:|
| Temporal IoU 동의 기준 | 0.5 |
| Visual–자막 동의 가산점 | 0.6 |
| KO–EN 동의 가산점 | 0.2 |
| Visual의 3경로 동의 추가 가산점 | 0.2 |
| 기본 순위 점수 | 1/rank |

위 설정은 실험적인 휴리스틱 규칙이며, 최적의 가중치로 검증된 값은 아니다.

---

## 3. 실험 환경 및 평가 데이터

### 3.1 사용 모델

**Visual 모델**
- Moment-DETR
- 기존 CLIP 영상 특징 사용
- 학습된 체크포인트: `scratch_clip_align_best_ORIGINAL.ckpt`
- 이번 실험에서는 추가 학습 없이 추론만 수행

**자막 검색 모델**
- Fine-tuned MiniLM
- AI-Hub 기반 한국어 데이터로 파인튜닝한 기존 모델 사용
- 임베딩 차원: 384
- 한국어·영어 검색어를 각각 입력해 후보 생성

### 3.2 평가 검색어

총 3개 한국어 영상, 6개 검색 사례를 사용하였다.

| 영상 | 검색 장면 | 정답 구간 |
|---|---|---|
| 두부 요리 | 두부 자르기 | 12.00~23.00초 |
| 두부 요리 | 두부 양념 준비 | 95.00~135.00초 |
| 주방 영상 | 가위로 소스 짜기 | 5.96~11.32초 |
| 주방 영상 | 쌀 씻기 | 29.00~32.00초 |
| 다이소 영상 | 여행용 파우치 | 3.08~8.40초 |
| 다이소 영상 | 여행용 시트 세제 | 13.00~18.00초 |

**주의:** 위 6개는 이전부터 반복적으로 실험해 온 개발용 검색 사례다. 독립적인 테스트 데이터가 아니므로 이 데이터에서의 개선만으로 일반화 성능을 주장할 수 없다.

### 3.3 평가 지표

- **Top-1 IoU:** 최종 1위 후보와 정답 구간의 시간적 겹침 정도
- **R1@0.5:** 최종 1위 후보의 IoU가 0.5 이상인 검색어 비율
- **R5@0.5:** 최종 Top-5에 IoU 0.5 이상인 후보가 존재하는 검색어 비율
- **Oracle@0.5:** 전체 후보 중 IoU 0.5 이상인 후보가 존재하는 검색어 비율

Oracle@0.5는 정답을 알고 가장 좋은 후보를 확인하는 진단용 지표이며, 실제 서비스 정확도를 의미하지 않는다.

---

## 4. Consensus V1·V2·V3 성능 비교

세 방식의 공정한 비교를 위해 **동일한 Visual·KO·EN 후보를 사용하고 재순위화 규칙만 변경**하였다.

| 평가 지표 | V1 | V2 | V3 |
|---|---:|---:|---:|
| 평균 Top-1 IoU | 0.4463 | 0.4463 | 0.4463 |
| 평균 Top-5 최고 IoU | 0.6166 | 0.6806 | 0.6806 |
| R1@0.5 | 50.0% | 50.0% | 50.0% |
| R5@0.5 | 66.7% | 83.3% | 83.3% |
| Oracle@0.5 | 100.0% | 100.0% | 100.0% |

### 4.1 주요 결과

1. V1·V2·V3의 Top-1 정확도는 모두 50.0%로 동일하였다.
2. V2·V3는 V1보다 R5@0.5가 16.7%p 높았다.
3. V3는 V2 대비 추가적인 성능 개선을 보여주지 못했다.
4. 전체 후보의 Oracle@0.5는 100%로, 6개 검색어 모두에서 IoU 0.5 이상의 후보가 하나 이상 존재하였다.
5. 그러나 실제 최종 1위로 올바른 후보를 선정한 사례는 3개뿐이었다.

**결론: 이번 실험에서 Consensus V3의 Top-1 성능 개선은 확인되지 않았다.**

다만 모든 검색어에서 유효한 후보가 존재했으므로, 이번 6개 사례에서는 후보 생성 실패보다 후보 선택 및 재순위화 문제가 직접적인 성능 제한 요인으로 확인되었다.

---

## 5. Consensus V3 실패 원인 분석

### 5.1 Top-1 실패 사례

6개 검색어 중 다음 3개에서 Top-1 IoU가 0.5 미만이었다.

| 검색 장면 | V3 Top-1 IoU | 가장 정확한 후보의 출처 | 최고 후보 IoU | 최고 후보의 V3 순위 |
|---|---:|---|---:|---:|
| 두부 자르기 | 0.0000 | MiniLM-EN | 0.5582 | 6위 |
| 두부 양념 준비 | 0.4350 | Visual | 0.6546 | 4위 |
| 다이소 시트 세제 | 0.1337 | MiniLM-KO | 0.7039 | 8위 |

가장 정확한 후보의 출처가 세 사례에서 모두 달랐다.

따라서 Visual, MiniLM-KO, MiniLM-EN 중 하나를 항상 우선하는 단순한 규칙으로는 전체 문제를 해결하기 어렵다.

### 5.2 원인 ① 검색 경로 간 동의 부족

현재 V3는 서로 다른 검색 경로의 후보가 Temporal IoU 0.5 이상으로 겹치면 가산점을 부여한다.

하지만 Top-1 실패 사례에서는 대부분의 후보에 `support_sources=[]`가 나타났다.

이는 해당 후보가 현재 V3의 기준을 충족하는 다른 검색 경로의 지지를 받지 못했다는 의미이다.

이 경우 동의 가산점이 발생하지 않아 기본 점수인 `1/rank`가 순위를 지배하였다.

**문제점:** 서로 다른 경로가 동일한 시간대를 지목하지 않으면 후보의 실제 품질을 구별할 추가적인 근거가 부족하다.

### 5.3 원인 ② 검색 경로 내부의 후보 순위 문제

**사례 A. 두부 자르기**

정답 구간: 12.00~23.00초

| 후보 | 시간 구간 | 정답 IoU | V3 순위 |
|---|---|---:|---:|
| Visual 1위 | 101.75~118.96초 | 0.0000 | 1위 |
| MiniLM-EN 1위 | 12.88~16.32초 | 0.3127 | 3위 |
| MiniLM-EN 2위 | 12.88~19.02초 | 0.5582 | 6위 |

영어 자막 검색이 유효한 후보를 생성했지만, 원래 2위였다는 이유로 최종 순위에서도 밀렸다.

**사례 B. 두부 양념 준비**

정답 구간: 95.00~135.00초

| 후보 | 시간 구간 | 정답 IoU | V3 순위 |
|---|---|---:|---:|
| Visual 1위 | 101.75~119.16초 | 0.4350 | 1위 |
| Visual 2위 | 90.21~151.32초 | 0.6546 | 4위 |

Visual 모델은 더 좋은 후보를 이미 생성했지만 기본 순위 점수 때문에 상대적으로 낮게 평가하였다.

**사례 C. 다이소 시트 세제**

정답 구간: 13.00~18.00초

| 후보 | 시간 구간 | 정답 IoU | V3 순위 |
|---|---|---:|---:|
| MiniLM-KO 1위 | 13.72~15.16초 | 0.2880 | 2위 |
| MiniLM-KO 2위 | 13.72~17.08초 | 0.6720 | 5위 |
| MiniLM-KO 3위 | 13.72~19.08초 | 0.7039 | 8위 |

동일한 시간대에서 생성된 여러 자막 후보가 서로 다른 길이를 가진다.

현재 V3는 이 후보들이 공유하는 구간 정보나 길이의 적절성을 충분히 반영하지 못한다.

단, 정답 구간이 길다고 해서 긴 후보가 항상 더 좋은 것은 아니다.

### 5.4 원인 ③ 서로 다른 모델 점수를 비교하기 어려움

Moment-DETR과 MiniLM은 서로 다른 기준으로 관련도 점수를 계산한다.

- Moment-DETR: 영상–텍스트 기반 모델 점수
- MiniLM: 문장 임베딩 사이의 코사인 유사도

따라서 서로 다른 경로에서 출력된 원점수를 직접 비교하거나 단순 합산하는 것은 적절하지 않을 수 있다.

후보 품질을 통합하려면 경로별 점수 분포, 순위, 시간적 일관성 등을 고려한 보정 또는 검증이 필요하다.

---

## 6. Consensus V3 실험 결론

### 확인된 사실

- V3는 이번 6개 사례에서 V2보다 Top-1 성능이 향상되지 않았다.
- 전체 후보에는 6개 모두 유효한 정답 후보가 존재하였다.
- 최종 Top-1은 3개 사례에서 실패하였다.
- 실패 사례에서 가장 정확한 후보의 출처는 Visual, KO, EN으로 각각 달랐다.
- 실패 사례의 후보 간 시간적 동의가 부족해 기본 순위 점수의 영향이 컸다.
- 개별 검색 경로 내부에서도 가장 높은 순위의 후보가 반드시 정답에 가장 가까운 것은 아니었다.

### 최종 판단

**Consensus V3는 현재 상태에서 Top-1 개선 목표를 달성하지 못하였다.**

다만 이 실험을 통해 후보 생성과 최종 후보 선정이 별개의 문제라는 점을 확인하였다.

기존 모델에서 이미 정답 후보를 생성하더라도, 재순위화 단계가 적절하게 작동하지 않으면 최종 검색 성능이 낮아질 수 있다.

이에 따라 향후 연구에서는 후보 품질 개선과 후보 재순위화 개선을 구분하여 진행한다.

---

## 7. 향후 개선 방향: Consensus V4

### 7.1 V4 개발 목적

Consensus V4는 V3의 단순한 시간적 동의 기반 점수 계산을 보완하고, **여러 후보 중 실제 검색어에 가장 적절한 구간을 최종 Top-1로 선택하는 것**을 목표로 한다.

V4는 아직 구현되지 않은 후속 실험 계획이다.

### 7.2 검토할 개선 방법

**① 검색 경로별 신뢰도 보정**

- Visual, KO, EN의 점수 범위 및 분포 분석
- 후보 순위와 관련도 점수의 관계 확인
- 서로 다른 모델의 원점수를 직접 합산하지 않고 적절한 보정 방식 검토
- 특정 검색 경로에 무조건 높은 고정 가중치를 부여하는 방식 지양

**② 검색 경로 내부의 후보 일관성 활용**

- 같은 시간대에 반복적으로 나타나는 후보들을 그룹화
- 서로 다른 길이로 생성된 후보들의 공통 구간 및 경계 분석
- 한 검색 경로 안에서도 관련 후보 간 유사성과 점수 차이를 활용할 수 있는지 검토

**③ 후보 시간 구간의 대표값 또는 보정 방식 실험**

- 같은 장면을 지목하는 후보 간 시작·종료 시간 비교
- 필요한 경우 후보 경계 보정 또는 대표 구간 생성
- 단순히 긴 후보를 우선하는 규칙은 사용하지 않음
- 실제 정답 정보 없이 적용할 수 있는 규칙만 추론 과정에 사용

**④ 서로 다른 검색 경로의 동의를 더 유연하게 활용**

- 기존 Temporal IoU 0.5 단일 기준의 한계 분석
- 후보 포함 관계 및 부분 겹침 정도 추가 검토
- Visual과 자막 검색의 동의, KO와 EN의 동의를 구분
- 단일 모델의 유사 결과가 과도한 표를 얻지 않도록 설계

### 7.3 V4 검증 방법

1. 기존 V1·V2·V3와 동일한 후보를 사용해 재순위화 방식만 비교한다.
2. R1@0.5, R5@0.5, 평균 Top-1 IoU를 평가한다.
3. 기존에 맞혔던 검색어가 V4에서 틀리게 되는지도 함께 확인한다.
4. 개발에 사용한 검색어와 독립적인 추가 검색어 또는 새로운 영상을 확보하여 검증한다.
5. 정답 구간을 최종 후보 선정 과정에 직접 사용하는 데이터 누수를 방지한다.

### 7.4 예상되는 한계

- 새로운 휴리스틱을 추가해도 Top-1 성능 향상은 보장되지 않는다.
- 기존 6개 검색어에만 맞춘 규칙은 다른 검색어에서 성능이 떨어질 수 있다.
- 모델별 관련도 점수는 직접 비교하기 어려울 수 있다.
- 검색 경로 간 동의가 거의 없는 경우 여전히 최종 후보 선정이 어려울 수 있다.

**따라서 V4는 개선 가능성을 검증할 후속 실험이지, 성능 향상이 확정된 모델은 아니다.**

---

## 8. Multilingual CLIP을 먼저 실험하기로 한 이유

### 8.1 기존 Visual 검색의 한계

현재 Moment-DETR은 기존 CLIP 기반 영상 특징과 영어 검색어의 텍스트 특징을 이용한다.

즉, 한국어 검색 서비스이지만 Visual 검색 경로에서는 영어 검색어를 사용하고 있다.

기존 영어 CLIP 기반 Visual 검색은 일부 사례에서 좋은 성능을 보였으나, 다른 사례에서는 정답과 무관한 시간 구간을 상위 후보로 반환하였다.

예를 들어 두부 자르기 검색에서는 Visual Top-3가 모두 정답 구간을 찾지 못하였다.

다만 이 결과만으로 실패 원인이 영어 검색어 사용이나 영어 CLIP에 있다고 단정할 수는 없다. 영상 특징, 모델 학습 데이터, 질의 표현 등 다른 요인도 영향을 미칠 수 있다.

### 8.2 Multilingual CLIP 실험 목적

Multilingual CLIP은 한국어를 포함한 여러 언어의 문장을 CLIP 계열 임베딩 공간으로 연결하는 방법이다.

이번 실험에서는 **한국어 검색어를 이용한 Visual 후보 생성이 기존 영어 검색어 기반 Visual 검색보다 나은지** 확인하고자 한다.

기존 Moment-DETR 모델 전체를 새로 학습하기보다는, 우선 한국어 텍스트 인코더의 연결 가능성을 검토한다.

### 8.3 Multilingual CLIP을 우선 실험하는 이유

**첫째, 후보 생성 성능을 별도로 개선할 수 있는지 확인하기 위해서다.**

현재 V4는 기존 후보를 재정렬하는 방향이다. 반면 Multilingual CLIP 실험은 기존 Visual 경로에서 생성하는 후보 자체를 바꿀 수 있는지 검증한다.

**둘째, 재순위화 규칙을 기존 소수 사례에 과도하게 맞추는 것을 피하기 위해서다.**

현재 평가 사례는 6개뿐이다. 이 데이터에 맞춰 V4의 가중치나 조건을 반복적으로 수정하면 과적합 위험이 있다.

**셋째, 기존 모델을 유지한 상태로 독립적인 비교 실험을 진행할 수 있기 때문이다.**

기존 영어 CLIP 기반 검색을 기준선으로 보존하고, 한국어 검색 방식을 별도 실험 경로로 구성해 후보의 품질을 비교할 계획이다.

**넷째, Multilingual CLIP이 유효하다면 이후 V4의 입력 후보를 개선하는 데 활용할 수 있기 때문이다.**

단, 한국어 Visual 후보가 개선되더라도 최종 Top-1이 자동으로 개선되는 것은 아니다. 후보 생성과 재순위화 성능은 각각 검증해야 한다.

### 8.4 Multilingual CLIP 적용 전 필수 확인 사항

Multilingual CLIP을 기존 Moment-DETR에 바로 연결할 수 있다고 가정해서는 안 된다.

다음 사항을 먼저 검증한다.

- 기존 영상 CLIP 특징과 Multilingual CLIP 텍스트 임베딩의 차원
- 두 임베딩이 실제로 호환되는 의미 공간에 있는지
- Moment-DETR이 요구하는 텍스트 특징의 형태 및 정규화 방식
- 기존 체크포인트의 학습 시 텍스트 특징과 새 특징의 분포 차이
- 새로운 텍스트 특징을 적용했을 때 기존 영어 검색 결과가 어떻게 달라지는지

**512차원 출력이라는 사실만으로 기존 Moment-DETR과의 호환성이 보장되는 것은 아니다.**

### 8.5 Multilingual CLIP 성능 평가 계획

1. 기존 Moment-DETR의 CLIP 텍스트 입력 구조 확인
2. 호환 가능한 Multilingual CLIP 후보 모델 조사
3. 기존 모델과 연결 가능한지 확인
4. 연결 가능할 경우 한국어 검색어로 Visual 후보 생성
5. 동일한 검색어에서 기존 영어 Visual과 한국어 Visual 비교
6. Visual 후보의 Top-1 IoU 및 후보 전체 최고 IoU 평가
7. 필요한 경우 Consensus 재순위화에 새로운 Visual 후보를 적용
8. 독립적인 추가 검색 사례로 성능 검증

호환성이 확보되지 않는 경우 기존 Moment-DETR의 텍스트 인코더를 무리하게 교체하지 않고, 별도의 검색 경로 구성 또는 다른 다국어 모델 활용 방안을 검토한다.

---

## 9. 향후 실험 진행 순서

| 순서 | 실험 | 목적 | 상태 |
|---|---|---|---|
| 1 | Consensus V1·V2·V3 비교 | 재순위화 성능 비교 | 완료 |
| 2 | Consensus V3 실패 원인 분석 | Top-1 실패 사례 및 원인 파악 | 완료 |
| 3 | Multilingual CLIP 호환성 점검 | 기존 Moment-DETR과 연결 가능성 확인 | 예정 |
| 4 | 한국어 Visual 검색 실험 | 기존 영어 Visual과 후보 품질 비교 | 예정 |
| 5 | Consensus와 Multilingual CLIP 결합 | 최종 Top-1 개선 여부 확인 | 예정 |
| 6 | Consensus V4 설계 및 실험 | 추가 후보 재순위화 개선 | 보류 |
| 7 | 추가 검색어·영상 평가 | 일반화 성능 검증 | 예정 |

V4는 폐기한 것이 아니라 **후속 개선 후보로 유지**한다.

Multilingual CLIP 적용 가능성과 효과를 먼저 확인한 뒤, 최종 Top-1 정확도가 충분하지 않을 경우 V4 개발을 진행한다.

---

## 10. 기존 결과 및 모델 보존 원칙

이번 실험은 기존에 개발한 Moment-DETR, Hybrid, Fine-tuned MiniLM, Fusion 및 Consensus V1·V2·V3 코드를 보존하는 것을 원칙으로 한다.

- 기존 체크포인트 수정 금지
- 기존 실험 JSON 덮어쓰기 금지
- GitHub `main` 브랜치 직접 변경 금지
- 새 모델·코드는 별도 실험 파일로 관리
- 실험 조건과 평가 결과 기록
- 개선된 결과만 선택적으로 보고하지 않고 실패 결과도 유지

### 주요 실험 결과 파일

**기존 6개 검색 사례 평가 목록**

`03_evaluation/consensus_v3_6cases_manifest.json`

**V1·V2·V3 동일 후보 성능 비교**

`03_evaluation/consensus_v1_v2_v3_6cases_step32.json`

**V3 실패 사례 상세 분석**

`03_evaluation/consensus_v3_failure_diagnosis_step33.json`

위 경로는 다음 실험 폴더를 기준으로 한다.

`04_PinPoint_ConsensusV3_MultilingualCLIP/`

---

## 11. 최종 결론

Consensus V3는 기존 V2 대비 Top-1 검색 정확도를 개선하지 못하였다.

6개 검색 사례에서 R1@0.5는 50.0%였으며, 정답 후보가 존재함에도 최종 1위 선정에 실패한 사례가 3개 확인되었다.

실패 원인은 검색 경로 간 시간적 동의 부족, 원래 후보 순위에 대한 높은 의존도, 검색 경로 내부의 후보 품질 차이를 충분히 반영하지 못하는 문제로 분석되었다.

이를 해결하기 위한 Consensus V4 개발은 후속 계획으로 유지하되, 우선 Multilingual CLIP을 활용한 한국어 Visual 후보 생성 가능성을 검토하기로 하였다.

**향후 목표는 기존 AI 모델을 무조건 교체하는 것이 아니라, 후보 생성 성능과 최종 재순위화 성능을 각각 검증하면서 PinPoint AI의 Top-1 검색 정확도를 개선하는 것이다.**

Multilingual CLIP 실험에서 개선 효과가 확인되지 않을 경우에도 결과를 보존하고, 기존 후보 기반 Consensus V4 개선 방향을 다시 검토할 예정이다.

# Multilingual CLIP 별도 실험
- 모델 로딩, 입력 형식 점검, 한국어 직접 검색과 기존 영어 검색 비교

In [39]:

from pathlib import Path
import inspect
import importlib
import importlib.util
import torch

print("===== 34단계. Multilingual CLIP 호환성 사전 점검 =====")

REPO = Path("/content/pinpoint-capstone-ai")

# 1. 기존 Moment-DETR 추론 모듈 확인
inference = importlib.import_module(
    "ai.features.run_moment_detr_inference"
)

expected_path = (
    REPO / "ai/features/run_moment_detr_inference.py"
).resolve()

actual_path = Path(inference.__file__).resolve()

print("\n[1] 현재 추론 코드 확인")
print("실제 코드:", actual_path)
print("예상 코드:", expected_path)

assert actual_path == expected_path, (
    "AI 실험 브랜치가 아닌 다른 코드를 불러왔습니다."
)

print("체크포인트:", inference.CHECKPOINT_PATH)
print("DEVICE:", inference.DEVICE)
print("CLIP_LEN:", getattr(inference, "CLIP_LEN", "없음"))

# 2. CLIP Feature Extractor 구현 확인
print("\n[2] CLIP Feature Extractor")

extractor_function = inference.get_feature_extractor

print("정의 파일:", inspect.getsourcefile(extractor_function))
print(inspect.getsource(extractor_function)[:5500])

# 3. 실제 Moment-DETR 텍스트 입력 부분 확인
print("\n[3] Moment-DETR의 텍스트 특징 생성 및 전달")

source = inspect.getsource(inference.predict_moments)
lines = source.splitlines()

keywords = (
    "encode_text",
    "text_feat",
    "query_feat",
    "query",
    "clip",
    "normalize",
    "model(",
)

selected = set()

for i, line in enumerate(lines):
    if any(k in line.lower() for k in keywords):
        for j in range(max(0, i - 2), min(len(lines), i + 4)):
            selected.add(j)

for j in sorted(selected):
    print(f"{j + 1:3d}: {lines[j]}")

# 4. 필요한 라이브러리 확인
print("\n[4] 다국어 모델 관련 라이브러리")

for package in [
    "sentence_transformers",
    "transformers",
    "open_clip",
]:
    spec = importlib.util.find_spec(package)
    print(
        package,
        "✅ 설치됨" if spec else "❌ 미설치"
    )

# 5. 기존 영상 CLIP 특징 차원
print("\n[5] 영상 Feature 형식")

FEATURE_DIR = Path(
    "/content/drive/MyDrive/pinpoint_capstone/"
    "02_Korean_Video_Test/clip_features"
)

for video_id in [
    "kr_tofu_001",
    "kr_kitchen_001",
    "kr_daiso_001",
]:
    path = FEATURE_DIR / f"{video_id}_clip_features.pt"

    assert path.is_file(), f"특징 파일 없음: {path}"

    features = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    if isinstance(features, torch.Tensor):
        print(
            video_id,
            "| shape:", tuple(features.shape),
            "| dtype:", features.dtype
        )
    elif isinstance(features, dict):
        print(video_id, "| 저장된 키:", list(features.keys()))
    else:
        print(video_id, "| 자료형:", type(features).__name__)

print("\n===== 34단계 완료 =====")

===== 34단계. Multilingual CLIP 호환성 사전 점검 =====

[1] 현재 추론 코드 확인
실제 코드: /content/pinpoint-capstone-ai/ai/features/run_moment_detr_inference.py
예상 코드: /content/pinpoint-capstone-ai/ai/features/run_moment_detr_inference.py
체크포인트: /content/drive/MyDrive/pinpoint_capstone/02_Korean_Video_Test/AI_Backup/scratch_clip_align_best_ORIGINAL.ckpt
DEVICE: cuda
CLIP_LEN: 2

[2] CLIP Feature Extractor
정의 파일: /content/pinpoint-capstone-ai/ai/features/run_moment_detr_inference.py
def get_feature_extractor():
    global _FEATURE_EXTRACTOR

    if _FEATURE_EXTRACTOR is None:
        print(f"[AI] Loading CLIP on {DEVICE}")

        _FEATURE_EXTRACTOR = ClipFeatureExtractor(
            framerate=1 / CLIP_LEN,
            size=224,
            centercrop=True,
            model_name_or_path="ViT-B/32",
            device=DEVICE
        )

    return _FEATURE_EXTRACTOR


[3] Moment-DETR의 텍스트 특징 생성 및 전달
  2:     video_id,
  3:     video_feature_path,
  4:     query,
  5:     video_duration=None,
  6:     top_

In [40]:

import torch
import inspect

print("===== 35단계. CLIP 텍스트 특징 실제 Shape 점검 =====")

print("\n[1] 기존 CLIP Feature Extractor 불러오기")

extractor = inference.get_feature_extractor()

print("Extractor 클래스:", type(extractor).__name__)

print("\n[2] 실제 영어 검색어 인코딩")

query = "cutting tofu with a knife"

with torch.no_grad():
    encoded = extractor.encode_text([query])

print("반환 자료형:", type(encoded).__name__)

if isinstance(encoded, torch.Tensor):
    print("전체 shape:", tuple(encoded.shape))
    print("dtype:", encoded.dtype)

elif isinstance(encoded, (list, tuple)):
    print("시퀀스 개수:", len(encoded))

    for i, item in enumerate(encoded):
        print(f"  [{i}] 자료형:", type(item).__name__)

        if hasattr(item, "shape"):
            print(f"  [{i}] shape:", tuple(item.shape))
            print(f"  [{i}] dtype:", getattr(item, "dtype", "없음"))

else:
    print("예상하지 못한 반환 형식:", type(encoded))

print("\n[3] Moment-DETR 모델 텍스트 입력 구조")

moment_model = inference.get_model()

print("모델 클래스:", type(moment_model).__name__)

try:
    signature = inspect.signature(moment_model.forward)
    print("forward 입력:", signature)
except Exception as e:
    print("forward 시그니처 확인 실패:", str(e))

print("\n[4] 텍스트 입력 관련 레이어 확인")

found = 0

for name, layer in moment_model.named_modules():
    name_lower = name.lower()

    if any(k in name_lower for k in [
        "input_txt",
        "txt_proj",
        "text_proj",
        "query_proj",
        "txt_input",
    ]):
        print(name, "→", type(layer).__name__)

        if isinstance(layer, torch.nn.Linear):
            print(
                "   입력 차원:",
                layer.in_features,
                "| 출력 차원:",
                layer.out_features
            )

        found += 1

        if found >= 20:
            break

if found == 0:
    print("이름으로 식별된 텍스트 투영 레이어가 없습니다.")

print("\n[5] CLIP 인코딩 함수의 구현 위치")

print("함수 위치:", inspect.getsourcefile(type(extractor).encode_text))

source = inspect.getsource(type(extractor).encode_text)

print(source[:5500])

print("\n===== 35단계 완료 =====")

===== 35단계. CLIP 텍스트 특징 실제 Shape 점검 =====

[1] 기존 CLIP Feature Extractor 불러오기
Extractor 클래스: ClipFeatureExtractor

[2] 실제 영어 검색어 인코딩
반환 자료형: list
시퀀스 개수: 1
  [0] 자료형: Tensor
  [0] shape: (7, 512)
  [0] dtype: torch.float16

[3] Moment-DETR 모델 텍스트 입력 구조
모델 클래스: MomentDETR
forward 입력: (src_txt, src_txt_mask, src_vid, src_vid_mask)

[4] 텍스트 입력 관련 레이어 확인
input_txt_proj → Sequential
input_txt_proj.0 → LinearLayer
input_txt_proj.0.LayerNorm → LayerNorm
input_txt_proj.0.net → Sequential
input_txt_proj.0.net.0 → Dropout
input_txt_proj.0.net.1 → Linear
   입력 차원: 512 | 출력 차원: 256
input_txt_proj.1 → LinearLayer
input_txt_proj.1.LayerNorm → LayerNorm
input_txt_proj.1.net → Sequential
input_txt_proj.1.net.0 → Dropout
input_txt_proj.1.net.1 → Linear
   입력 차원: 256 | 출력 차원: 256

[5] CLIP 인코딩 함수의 구현 위치
함수 위치: /usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py
    @torch.no_grad()
    def encode_text(self, text_list, bsz=60):
        n_text = len(text_list)
        n_batch = int(math.cei

In [41]:

from pathlib import Path
import json
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

print("===== 36단계. Multilingual CLIP 한국어 Visual 검색 =====")

DRIVE = Path("/content/drive/MyDrive/pinpoint_capstone")

BASE = (
    DRIVE /
    "04_PinPoint_ConsensusV3_MultilingualCLIP"
)

CACHE_DIR = BASE / "04_models/huggingface_cache"
RESULT_DIR = BASE / "02_results"

FEATURE_PATH = (
    DRIVE / "02_Korean_Video_Test/"
    "clip_features/kr_tofu_001_clip_features.pt"
)

OUTPUT_PATH = (
    RESULT_DIR / "tofu_multilingual_clip_frame_search_step36.json"
)

MODEL_NAME = (
    "sentence-transformers/"
    "clip-ViT-B-32-multilingual-v1"
)

QUERY_KO = "두부를 칼로 자르는 장면"

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"기존 결과는 덮어쓰지 않습니다: {OUTPUT_PATH}"
    )

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("\n[1] Multilingual CLIP 다운로드 및 로딩")
print("모델:", MODEL_NAME)

# 기존 Moment-DETR의 GPU 메모리를 보호하기 위해 CPU 사용
multilingual_clip = SentenceTransformer(
    MODEL_NAME,
    device="cpu",
    cache_folder=str(CACHE_DIR)
)

multilingual_clip.eval()

print("✅ 모델 로딩 성공")
print(
    "문장 임베딩 차원:",
    multilingual_clip.get_embedding_dimension()
)

print("\n[2] 한국어 검색어 인코딩")

query_embedding = multilingual_clip.encode(
    [QUERY_KO],
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False
).float().cpu()

print("검색어:", QUERY_KO)
print("임베딩 shape:", tuple(query_embedding.shape))

print("\n[3] 기존 영상 CLIP 특징 불러오기")

video_features = torch.load(
    FEATURE_PATH,
    map_location="cpu",
    weights_only=True
)

assert isinstance(video_features, torch.Tensor)

video_features = video_features.float()

print("영상 Feature shape:", tuple(video_features.shape))

if (
    query_embedding.ndim != 2
    or video_features.ndim != 2
    or query_embedding.shape[1] != video_features.shape[1]
):
    raise ValueError(
        "텍스트와 영상 특징의 차원이 맞지 않습니다."
    )

print("\n[4] 프레임별 코사인 유사도 계산")

video_normalized = F.normalize(
    video_features,
    p=2,
    dim=-1
)

similarities = (
    query_embedding @ video_normalized.T
).squeeze(0)

top_k = min(5, len(similarities))
scores, indices = torch.topk(
    similarities,
    k=top_k
)

results = []

for rank, (idx, score) in enumerate(
    zip(indices.tolist(), scores.tolist()), 1
):
    # 기존 특징은 약 2초 간격으로 추출됨.
    # 실제 장면의 정확한 시작·종료 시각은 아님.
    approx_time = idx * 2.0

    item = {
        "rank": rank,
        "feature_index": idx,
        "approx_time_seconds": approx_time,
        "cosine_similarity": score
    }
    results.append(item)

    print(
        f"{rank}위 | "
        f"특징 인덱스={idx} | "
        f"대략 {approx_time:.2f}초 | "
        f"유사도={score:.4f}"
    )

print("\n[5] 실험 결과 저장")

output = {
    "experiment": "Multilingual CLIP frame-level pilot",
    "model": MODEL_NAME,
    "video_id": "kr_tofu_001",
    "query_ko": QUERY_KO,
    "video_feature_shape": list(video_features.shape),
    "text_embedding_shape": list(query_embedding.shape),
    "time_step_seconds": 2.0,
    "note": (
        "Frame-level cosine similarity pilot. "
        "Approximate times are feature indices multiplied by 2 seconds. "
        "Not Moment-DETR predictions or temporal IoU evaluation."
    ),
    "top_frames": results
}

with OUTPUT_PATH.open("x", encoding="utf-8") as f:
    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ 저장 완료:", OUTPUT_PATH)
print("\n===== 36단계 완료 =====")

===== 36단계. Multilingual CLIP 한국어 Visual 검색 =====

[1] Multilingual CLIP 다운로드 및 로딩
모델: sentence-transformers/clip-ViT-B-32-multilingual-v1


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.65k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.57MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

✅ 모델 로딩 성공
문장 임베딩 차원: 512

[2] 한국어 검색어 인코딩
검색어: 두부를 칼로 자르는 장면
임베딩 shape: (1, 512)

[3] 기존 영상 CLIP 특징 불러오기
영상 Feature shape: (83, 512)

[4] 프레임별 코사인 유사도 계산
1위 | 특징 인덱스=8 | 대략 16.00초 | 유사도=0.2477
2위 | 특징 인덱스=67 | 대략 134.00초 | 유사도=0.2454
3위 | 특징 인덱스=10 | 대략 20.00초 | 유사도=0.2446
4위 | 특징 인덱스=31 | 대략 62.00초 | 유사도=0.2438
5위 | 특징 인덱스=48 | 대략 96.00초 | 유사도=0.2435

[5] 실험 결과 저장
✅ 저장 완료: /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/02_results/tofu_multilingual_clip_frame_search_step36.json

===== 36단계 완료 =====


In [42]:

from pathlib import Path
import json
import torch
import torch.nn.functional as F
import numpy as np

print("===== 37단계. Multilingual CLIP 6개 검색어 평가 =====")

DRIVE = Path("/content/drive/MyDrive/pinpoint_capstone")
BASE = DRIVE / "04_PinPoint_ConsensusV3_MultilingualCLIP"

MANIFEST_PATH = (
    BASE / "03_evaluation/consensus_v3_6cases_manifest.json"
)
RESULT_PATH = (
    BASE / "03_evaluation/"
    "multilingual_clip_6cases_frame_evaluation_step37.json"
)
FEATURE_DIR = DRIVE / "02_Korean_Video_Test/clip_features"

if RESULT_PATH.exists():
    raise FileExistsError(
        f"기존 결과 보호: {RESULT_PATH}"
    )

if "multilingual_clip" not in globals():
    raise RuntimeError(
        "36단계에서 로딩한 Multilingual CLIP이 없습니다."
    )

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

cases = manifest["cases"]
video_info = manifest["video_info"]

print("\n[1] 실험 설정")
print("검색어 수:", len(cases))
print("모델:", "clip-ViT-B-32-multilingual-v1")
print("평가 방식: 프레임 위치 기반 Top-1 / Top-5")
print("참고: 구간 IoU 평가가 아님")

# 검색어를 한 번에 인코딩
queries = [case["query_ko"] for case in cases]

text_embeddings = multilingual_clip.encode(
    queries,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False
).float().cpu()

assert text_embeddings.shape == (6, 512)

# 영상 특징은 영상당 한 번씩만 읽기
feature_cache = {}

for video_id, info in video_info.items():
    path = FEATURE_DIR / info["feature"]

    if not path.is_file():
        raise FileNotFoundError(path)

    features = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    ).float()

    assert features.ndim == 2
    assert features.shape[1] == 512

    feature_cache[video_id] = F.normalize(
        features,
        p=2,
        dim=-1
    )

print("✅ 검색어 임베딩 및 영상 특징 준비 완료")

# 모든 후보 생성 후에만 정답으로 평가
all_results = []

for index, case in enumerate(cases):
    name = case["name"]
    video_id = case["video_id"]

    text_vector = text_embeddings[index]
    video_vectors = feature_cache[video_id]

    similarities = video_vectors @ text_vector

    k = min(5, len(similarities))
    scores, indices = torch.topk(similarities, k=k)

    top_frames = []

    for rank, (frame_index, score) in enumerate(
        zip(indices.tolist(), scores.tolist()), 1
    ):
        top_frames.append({
            "rank": rank,
            "feature_index": frame_index,
            "approx_time_seconds": frame_index * 2.0,
            "cosine_similarity": float(score)
        })

    # 정답은 후보 순위를 결정한 다음 사용
    gt = case["ground_truth"]
    gt_start = float(gt["start_time"])
    gt_end = float(gt["end_time"])

    for frame in top_frames:
        t = frame["approx_time_seconds"]
        frame["point_in_gt"] = (
            gt_start <= t <= gt_end
        )

    top1_hit = int(top_frames[0]["point_in_gt"])
    top5_hit = int(any(
        frame["point_in_gt"] for frame in top_frames
    ))

    all_results.append({
        "name": name,
        "video_id": video_id,
        "query_ko": case["query_ko"],
        "ground_truth": {
            "start_time": gt_start,
            "end_time": gt_end
        },
        "top1_point_hit": top1_hit,
        "top5_point_hit": top5_hit,
        "top_frames": top_frames
    })

    print("\n" + "=" * 60)
    print(f"[{index+1}/6] {name}")
    print(f"정답: {gt_start:.2f}~{gt_end:.2f}초")

    for frame in top_frames:
        status = (
            "✅ 정답 시간대"
            if frame["point_in_gt"]
            else "❌ 다른 시간대"
        )

        print(
            f"{frame['rank']}위 | "
            f"{frame['approx_time_seconds']:.2f}초 | "
            f"sim={frame['cosine_similarity']:.4f} | "
            f"{status}"
        )

print("\n[2] 전체 요약")

top1_count = sum(
    row["top1_point_hit"] for row in all_results
)
top5_count = sum(
    row["top5_point_hit"] for row in all_results
)

n = len(all_results)

print(
    f"Top-1 Point Hit: {top1_count}/{n} "
    f"({top1_count/n*100:.1f}%)"
)

print(
    f"Top-5 Point Hit: {top5_count}/{n} "
    f"({top5_count/n*100:.1f}%)"
)

print("\n[3] Google Drive 저장")

output = {
    "experiment": "Multilingual CLIP Korean Frame Retrieval",
    "model": (
        "sentence-transformers/"
        "clip-ViT-B-32-multilingual-v1"
    ),
    "case_count": n,
    "time_step_seconds": 2.0,
    "metric_warning": (
        "Point-in-ground-truth at approximate feature "
        "timestamps. Not temporal IoU, R1@0.5, "
        "or Moment-DETR segment evaluation."
    ),
    "summary": {
        "top1_point_hit_count": top1_count,
        "top5_point_hit_count": top5_count,
        "top1_point_hit_rate": top1_count / n,
        "top5_point_hit_rate": top5_count / n
    },
    "results": all_results
}

RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)

with RESULT_PATH.open("x", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ 저장 완료:", RESULT_PATH)
print("\n===== 37단계 완료 =====")

===== 37단계. Multilingual CLIP 6개 검색어 평가 =====

[1] 실험 설정
검색어 수: 6
모델: clip-ViT-B-32-multilingual-v1
평가 방식: 프레임 위치 기반 Top-1 / Top-5
참고: 구간 IoU 평가가 아님
✅ 검색어 임베딩 및 영상 특징 준비 완료

[1/6] 두부 자르기
정답: 12.00~23.00초
1위 | 16.00초 | sim=0.2477 | ✅ 정답 시간대
2위 | 134.00초 | sim=0.2454 | ❌ 다른 시간대
3위 | 20.00초 | sim=0.2446 | ✅ 정답 시간대
4위 | 62.00초 | sim=0.2438 | ❌ 다른 시간대
5위 | 96.00초 | sim=0.2435 | ❌ 다른 시간대

[2/6] 두부 양념 준비
정답: 95.00~135.00초
1위 | 20.00초 | sim=0.2402 | ❌ 다른 시간대
2위 | 134.00초 | sim=0.2398 | ✅ 정답 시간대
3위 | 18.00초 | sim=0.2387 | ❌ 다른 시간대
4위 | 58.00초 | sim=0.2383 | ❌ 다른 시간대
5위 | 52.00초 | sim=0.2357 | ❌ 다른 시간대

[3/6] 가위로 소스 짜기
정답: 5.96~11.32초
1위 | 36.00초 | sim=0.2548 | ❌ 다른 시간대
2위 | 18.00초 | sim=0.2389 | ❌ 다른 시간대
3위 | 28.00초 | sim=0.2383 | ❌ 다른 시간대
4위 | 34.00초 | sim=0.2352 | ❌ 다른 시간대
5위 | 16.00초 | sim=0.2340 | ❌ 다른 시간대

[4/6] 쌀 씻기
정답: 29.00~32.00초
1위 | 36.00초 | sim=0.2845 | ❌ 다른 시간대
2위 | 22.00초 | sim=0.2784 | ❌ 다른 시간대
3위 | 28.00초 | sim=0.2686 | ❌ 다른 시간대
4위 | 18.00초 | sim=0.2681 | ❌ 다른 시간대
5위 | 30.00초 | 

In [43]:

import torch
import inspect

print("===== 38단계. 기존 영어 CLIP 문장 임베딩 구조 확인 =====")

extractor = inference.get_feature_extractor()
clip_model = extractor.clip_extractor

query = "cutting tofu with a knife"

print("\n[1] CLIP 모델 확인")
print("모델 클래스:", type(clip_model).__name__)
print("모델 모듈:", type(clip_model).__module__)

print("\n[2] 토큰화 및 원본 인코딩")

tokens = extractor.tokenizer(
    [query],
    context_length=77
).to(extractor.device)

print("토큰 shape:", tuple(tokens.shape))
print("유효 토큰 수:", int((tokens != 0).sum()))

with torch.no_grad():
    raw_output = clip_model.encode_text(tokens)

print("\n[3] encode_text 반환 구조")
print("자료형:", type(raw_output).__name__)

if isinstance(raw_output, dict):
    for key, value in raw_output.items():
        if torch.is_tensor(value):
            print(
                f"{key}: shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(f"{key}: {type(value).__name__}")

elif torch.is_tensor(raw_output):
    print("shape:", tuple(raw_output.shape))

else:
    print("반환값 자료형:", type(raw_output))

print("\n[4] 원본 CLIP encode_text 구현")

# 래퍼 함수 대신 실제 원본 함수의 코드를 확인
method = clip_model.encode_text

while hasattr(method, "__wrapped__"):
    method = method.__wrapped__

try:
    source = inspect.getsource(method)
    print(source[:6500])
except (OSError, TypeError) as e:
    print("소스 확인 불가:", str(e))

print("\n===== 38단계 완료 =====")

===== 38단계. 기존 영어 CLIP 문장 임베딩 구조 확인 =====

[1] CLIP 모델 확인
모델 클래스: CLIP
모델 모듈: run_on_video.clip.model

[2] 토큰화 및 원본 인코딩
토큰 shape: (1, 77)
유효 토큰 수: 7

[3] encode_text 반환 구조
자료형: dict
last_hidden_state: shape=(1, 77, 512), dtype=torch.float16
pooler_output: shape=(1, 512), dtype=torch.float16

[4] 원본 CLIP encode_text 구현
    def encode_text(self, text):
        x = self.token_embedding(text).type(self.dtype)  # [batch_size, n_ctx, d_model]

        x = x + self.positional_embedding.type(self.dtype)
        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD
        x = self.ln_final(x).type(self.dtype)

        # x.shape = [batch_size, n_ctx, transformer.width]
        # take features from the eot embedding (eot_token is the highest number in each sequence)
        eos_x = x[torch.arange(x.shape[0]), text.argmax(dim=-1)] @ self.text_projection

        return dict(last_hidden_state=x, pooler_output=eos_x)


===== 38단계 완료 =====


In [44]:

from pathlib import Path
import json
import torch
import torch.nn.functional as F

print("===== 39단계. 영어 CLIP vs Multilingual CLIP =====")

DRIVE = Path("/content/drive/MyDrive/pinpoint_capstone")
BASE = DRIVE / "04_PinPoint_ConsensusV3_MultilingualCLIP"
OLD = DRIVE / "02_Korean_Video_Test"

MANIFEST_PATH = (
    BASE / "03_evaluation/consensus_v3_6cases_manifest.json"
)
MULTI_PATH = (
    BASE / "03_evaluation/"
    "multilingual_clip_6cases_frame_evaluation_step37.json"
)
OUTPUT_PATH = (
    BASE / "03_evaluation/"
    "english_vs_multilingual_clip_6cases_step39.json"
)

for path in [MANIFEST_PATH, MULTI_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"기존 결과 보호: {OUTPUT_PATH}"
    )

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

with MULTI_PATH.open("r", encoding="utf-8") as f:
    multi_data = json.load(f)

cases = manifest["cases"]
video_info = manifest["video_info"]
multi_rows = multi_data["results"]

assert len(cases) == len(multi_rows) == 6

extractor = inference.get_feature_extractor()
clip_model = extractor.clip_extractor

# 기존 CLIP 문장 전체 임베딩 생성
print("\n[1] 영어 CLIP 문장 임베딩 생성")

english_queries = [case["query_en"] for case in cases]

tokens = extractor.tokenizer(
    english_queries,
    context_length=77
).to(extractor.device)

with torch.no_grad():
    output = clip_model.encode_text(tokens)
    english_embeddings = (
        output["pooler_output"]
        .float()
        .cpu()
    )

english_embeddings = F.normalize(
    english_embeddings,
    p=2,
    dim=-1
)

print("영어 문장 임베딩:", tuple(english_embeddings.shape))
assert english_embeddings.shape == (6, 512)

# 영상 특징 준비
print("\n[2] 영상 특징 불러오기")

feature_cache = {}

for video_id, info in video_info.items():
    path = OLD / "clip_features" / info["feature"]

    features = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    ).float()

    assert features.ndim == 2
    assert features.shape[1] == 512

    feature_cache[video_id] = F.normalize(
        features,
        p=2,
        dim=-1
    )

print("✅ 영상 특징 준비 완료")

# 영어 CLIP 후보를 생성한 뒤 정답으로 평가
print("\n[3] 검색어별 비교")

comparisons = []

for i, case in enumerate(cases):
    vid = case["video_id"]
    sims = feature_cache[vid] @ english_embeddings[i]

    k = min(5, len(sims))
    scores, indices = torch.topk(sims, k=k)

    english_frames = []

    for rank, (idx, score) in enumerate(
        zip(indices.tolist(), scores.tolist()), 1
    ):
        english_frames.append({
            "rank": rank,
            "feature_index": idx,
            "approx_time_seconds": idx * 2.0,
            "cosine_similarity": float(score)
        })

    gt = case["ground_truth"]
    gs = float(gt["start_time"])
    ge = float(gt["end_time"])

    for row in english_frames:
        t = row["approx_time_seconds"]
        row["point_in_gt"] = gs <= t <= ge

    english_top1 = int(english_frames[0]["point_in_gt"])
    english_top5 = int(any(
        r["point_in_gt"] for r in english_frames
    ))

    multi_row = multi_rows[i]

    assert multi_row["name"] == case["name"]
    assert multi_row["video_id"] == vid

    multi_top1 = multi_row["top1_point_hit"]
    multi_top5 = multi_row["top5_point_hit"]

    comparisons.append({
        "name": case["name"],
        "video_id": vid,
        "query_en": case["query_en"],
        "query_ko": case["query_ko"],
        "ground_truth": gt,
        "english_clip": {
            "top1_point_hit": english_top1,
            "top5_point_hit": english_top5,
            "top_frames": english_frames
        },
        "multilingual_clip": {
            "top1_point_hit": multi_top1,
            "top5_point_hit": multi_top5,
            "top_frames": multi_row["top_frames"]
        }
    })

    print("\n" + "=" * 56)
    print(case["name"])
    print(
        f"English CLIP : "
        f"Top1={english_top1} / Top5={english_top5} "
        f"| 1위={english_frames[0]['approx_time_seconds']:.2f}초"
    )
    print(
        f"Multilingual : "
        f"Top1={multi_top1} / Top5={multi_top5} "
        f"| 1위={multi_row['top_frames'][0]['approx_time_seconds']:.2f}초"
    )

print("\n[4] 전체 성능 비교")

n = len(comparisons)

summary = {}

for key in ["english_clip", "multilingual_clip"]:
    r1 = sum(c[key]["top1_point_hit"] for c in comparisons)
    r5 = sum(c[key]["top5_point_hit"] for c in comparisons)

    summary[key] = {
        "top1_point_hit_count": r1,
        "top5_point_hit_count": r5,
        "top1_point_hit_rate": r1 / n,
        "top5_point_hit_rate": r5 / n
    }

    print(f"\n[{key}]")
    print(f"Top-1 Point Hit: {r1}/{n} ({r1/n*100:.1f}%)")
    print(f"Top-5 Point Hit: {r5}/{n} ({r5/n*100:.1f}%)")

print("\n[5] 결과 저장")

result = {
    "experiment": "English CLIP vs Multilingual CLIP Frame Retrieval",
    "case_count": n,
    "comparison_note": (
        "English CLIP uses English queries and pooler_output. "
        "Multilingual CLIP uses Korean queries and sentence embeddings. "
        "Both use the same video frame features and point-hit metric. "
        "This is not Moment-DETR temporal localization evaluation."
    ),
    "summary": summary,
    "results": comparisons
}

with OUTPUT_PATH.open("x", encoding="utf-8") as f:
    json.dump(
        result,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ 저장 완료:", OUTPUT_PATH)
print("\n===== 39단계 완료 =====")

===== 39단계. 영어 CLIP vs Multilingual CLIP =====

[1] 영어 CLIP 문장 임베딩 생성
영어 문장 임베딩: (6, 512)

[2] 영상 특징 불러오기
✅ 영상 특징 준비 완료

[3] 검색어별 비교

두부 자르기
English CLIP : Top1=0 / Top5=1 | 1위=52.00초
Multilingual : Top1=1 / Top5=1 | 1위=16.00초

두부 양념 준비
English CLIP : Top1=0 / Top5=0 | 1위=52.00초
Multilingual : Top1=0 / Top5=1 | 1위=20.00초

가위로 소스 짜기
English CLIP : Top1=1 / Top5=1 | 1위=8.00초
Multilingual : Top1=0 / Top5=0 | 1위=36.00초

쌀 씻기
English CLIP : Top1=0 / Top5=1 | 1위=36.00초
Multilingual : Top1=0 / Top5=1 | 1위=36.00초

다이소 파우치
English CLIP : Top1=1 / Top5=1 | 1위=6.00초
Multilingual : Top1=1 / Top5=1 | 1위=6.00초

다이소 시트 세제
English CLIP : Top1=1 / Top5=1 | 1위=14.00초
Multilingual : Top1=0 / Top5=1 | 1위=30.00초

[4] 전체 성능 비교

[english_clip]
Top-1 Point Hit: 3/6 (50.0%)
Top-5 Point Hit: 5/6 (83.3%)

[multilingual_clip]
Top-1 Point Hit: 2/6 (33.3%)
Top-5 Point Hit: 5/6 (83.3%)

[5] 결과 저장
✅ 저장 완료: /content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/03_evaluation/english_vs_multi

# 기억할 사항
# PinPoint AI — Multilingual CLIP 실험 결과 및 향후 개선 계획

## 1. 실험 개요

### 1.1 실험 배경

PinPoint AI는 사용자가 자연어로 원하는 장면을 검색하면, 영상 내부에서 관련 장면의 시간 구간을 찾아 반환하는 서비스이다.

기존 AI 검색 파이프라인은 다음 세 가지 검색 경로를 활용한다.

1. **Visual Moment-DETR:** 영어 검색어와 영상 CLIP 특징을 이용한 시간 구간 탐색
2. **Fine-tuned MiniLM-KO:** 한국어 검색어와 한국어 자막 간 의미적 유사도 검색
3. **Fine-tuned MiniLM-EN:** 영어 검색어와 한국어 자막 간 의미적 유사도 검색

이후 각 검색 경로에서 생성한 후보를 결합하여 최종 검색 결과를 결정한다.

기존 Consensus V1·V2·V3 실험에서는 6개 검색어를 기준으로 다음 결과를 확인하였다.

| 지표 | Consensus V3 |
|---|---:|
| R1@0.5 | 50.0% |
| R5@0.5 | 83.3% |
| Oracle@0.5 | 100.0% |

전체 후보에는 모든 검색어의 정답 후보가 존재했지만, 실제 최종 1위 선정에 성공한 사례는 3개뿐이었다.

이에 따라 재순위화 방식을 개선하는 Consensus V4가 후속 과제로 제안되었다.

그러나 기존 6개 검색어에 맞춰 재순위화 규칙을 반복적으로 수정하면 과적합이 발생할 가능성이 있기 때문에, V4 개발에 앞서 **새로운 Visual 검색 경로를 통해 후보 생성 성능 자체를 개선할 수 있는지** 검토하기로 하였다.

이를 위해 Multilingual CLIP 실험을 진행하였다.

### 1.2 실험 목적

이번 실험의 주요 목적은 다음과 같다.

- 기존 영어 CLIP이 아닌 한국어 Multilingual CLIP으로 영상 내 관련 프레임을 검색할 수 있는지 확인
- Multilingual CLIP과 기존 CLIP의 영상 검색 성능 비교
- 영어·한국어 검색 경로의 상호보완 가능성 분석
- 기존 Moment-DETR과 연결 가능한지 기술적 호환성 검토
- 향후 Consensus 및 PinPoint AI 검색 시스템에 적용할 가치가 있는지 판단

**본 실험은 기존 Moment-DETR 모델을 교체하거나 재학습한 실험이 아니라, Multilingual CLIP을 별도의 Visual 검색 경로로 활용할 수 있는지 확인한 초기 탐색 실험이다.**

---

## 2. 사용 모델 및 실험 환경

### 2.1 기존 영어 CLIP

- 모델: CLIP ViT-B/32
- 사용 언어: 영어
- 영상 특징 차원: 512
- 문장 임베딩 차원: 512
- 기존 Moment-DETR의 영상·텍스트 특징 생성에 사용되는 CLIP 구현 활용

기존 CLIP의 `encode_text()`는 다음 두 가지 출력을 제공한다.

| 출력 | Shape | 의미 |
|---|---|---|
| `last_hidden_state` | `(1, 77, 512)` | 토큰별 텍스트 특징 |
| `pooler_output` | `(1, 512)` | 문장 전체 임베딩 |

기존 Moment-DETR은 `last_hidden_state`에서 유효한 토큰에 해당하는 특징을 사용한다.

실제 검색어 `cutting tofu with a knife`의 유효 토큰 특징은 `(7, 512)` 형태로 확인되었다.

반면 이번 프레임 검색 비교에서는 문장 전체를 나타내는 `pooler_output`을 사용하였다.

### 2.2 Multilingual CLIP

- 모델: `sentence-transformers/clip-ViT-B-32-multilingual-v1`
- 사용 언어: 한국어
- 문장 임베딩 차원: 512
- SentenceTransformer 방식으로 모델 로딩
- 기존 CLIP 영상 특징과 코사인 유사도 계산

실제 한국어 검색어 `두부를 칼로 자르는 장면`에 대한 출력 형태는 `(1, 512)`로 확인되었다.

해당 모델은 다국어 문장을 CLIP 계열 임베딩 공간으로 연결하기 위한 모델이다.

다만 **출력 차원이 512로 동일하다는 사실만으로 기존 Moment-DETR과 직접 호환된다는 의미는 아니다.**

### 2.3 사용 영상 및 특징

기존 한국어 영상 테스트에서 생성했던 CLIP 특징을 재사용하였다.

| 영상 | Feature Shape | 특징 추출 간격 |
|---|---|---|
| 두부 요리 | `(83, 512)` | 약 2초 |
| 주방 영상 | `(19, 512)` | 약 2초 |
| 다이소 영상 | `(16, 512)` | 약 2초 |

기존 특징을 그대로 사용하므로 새로운 영상 특징 추출이나 Moment-DETR 재학습은 진행하지 않았다.

### 2.4 실험 환경

- Google Colab
- Python 및 PyTorch
- SentenceTransformers
- 기존 PinPoint AI 브랜치의 CLIP 구현
- 기존 Google Drive 영상 특징 및 평가 데이터

기존 Moment-DETR 체크포인트, Fine-tuned MiniLM, Fusion, Consensus V1·V2·V3 코드는 수정하지 않았다.

---

## 3. Multilingual CLIP 검색 구조

### 3.1 기존 Moment-DETR 기반 검색

기존 Moment-DETR은 다음 구조로 동작한다.

```text
영어 검색어
    ↓
CLIP ViT-B/32 Text Encoder
    ↓
토큰별 텍스트 특징
    ↓
기존 영상 CLIP 특징과 함께 Moment-DETR 입력
    ↓
관련 영상 구간 예측
    ↓
start_time / end_time 반환
```

Moment-DETR은 영상과 텍스트 특징을 입력받아 **시간 구간 자체를 예측하는 모델**이다.

### 3.2 이번 Multilingual CLIP 실험 구조

```text
한국어 검색어
    ↓
Multilingual CLIP Text Encoder
    ↓
512차원 문장 임베딩
    ↓
기존 영상 CLIP 특징과 코사인 유사도 계산
    ↓
프레임별 관련도 점수
    ↓
상위 Top-5 프레임 반환
```

각 영상 특징은 약 2초 간격으로 추출되었으므로, 프레임 위치를 다음과 같이 근사하였다.

```text
대략적인 프레임 시각 = Feature Index × 2초
```

예를 들어 Feature Index가 8이라면 약 16초 지점으로 해석하였다.

### 3.3 이번 실험의 핵심 차이

**Moment-DETR 구간 검색과 CLIP 프레임 검색은 서로 다른 작업이다.**

| 구분 | Moment-DETR | 이번 CLIP 실험 |
|---|---|---|
| 입력 | 영상 특징 + 텍스트 특징 | 영상 특징 + 문장 임베딩 |
| 출력 | 시작·종료 시간 구간 | 관련 프레임 위치 |
| 평가 | Temporal IoU | Point Hit |
| 최종 서비스 구간 반환 | 가능 | 별도 구간 생성 필요 |

따라서 이번 Multilingual CLIP의 프레임 검색 결과를 기존 Moment-DETR의 R1@0.5와 직접 비교하지 않았다.

---

## 4. 평가 데이터 및 지표

### 4.1 평가 검색어

이전 Consensus 실험에서 사용한 동일한 6개 검색 사례를 활용하였다.

| 영상 | 검색 장면 | 정답 구간 |
|---|---|---|
| 두부 | 두부 자르기 | 12.00~23.00초 |
| 두부 | 두부 양념 준비 | 95.00~135.00초 |
| 주방 | 가위로 소스 짜기 | 5.96~11.32초 |
| 주방 | 쌀 씻기 | 29.00~32.00초 |
| 다이소 | 여행용 화장품 파우치 | 3.08~8.40초 |
| 다이소 | 여행용 시트 세제 | 13.00~18.00초 |

영어 CLIP에는 기존 영어 검색어를 입력하고, Multilingual CLIP에는 대응하는 한국어 검색어를 입력하였다.

이 비교에서는 **검색어 언어와 텍스트 인코더가 동시에 달라지므로**, 관찰된 성능 차이를 인코더 자체의 효과로만 해석할 수는 없다.

또한 평가 검색어 6개는 이전부터 사용한 개발 사례이며 독립적인 테스트셋은 아니다.

### 4.2 Top-1 Point Hit

프레임 검색 결과 1위가 정답 시간 구간 안에 위치하는지 확인한다.

예를 들어 정답이 12~23초이고 예측 프레임이 16초라면 성공으로 평가한다.

### 4.3 Top-5 Point Hit

검색된 상위 5개 프레임 중 하나라도 정답 시간 구간 안에 위치하면 성공으로 평가한다.

### 4.4 평가 시 주의사항

Point Hit은 프레임 하나가 정답 구간 안에 있는지를 판단한다.

따라서 다음과 같은 한계가 있다.

- 장면의 정확한 시작·종료 시각을 평가하지 못함
- 정답 구간 길이에 따라 성공 가능성이 달라질 수 있음
- 약 2초 간격의 특징 추출로 인해 짧은 장면을 놓칠 수 있음
- 실제 프레임 타임스탬프 대신 특징 인덱스로 계산한 근사 시각 사용
- Top-5에 정답 프레임이 존재해도 최종 1위 선택 성공을 의미하지 않음

**이번 지표는 프레임 검색을 검증하기 위한 초기 평가 지표이며, 기존 Moment-DETR의 Temporal IoU 기반 검색 정확도와 다르다.**

---

## 5. Multilingual CLIP 단독 실험 결과

### 5.1 두부 자르기 첫 테스트

검색어: `두부를 칼로 자르는 장면`

정답 구간: 12~23초

| 순위 | 검색 프레임 | 코사인 유사도 | 정답 구간 포함 |
|---|---:|---:|---|
| 1위 | 16초 | 0.2477 | 성공 |
| 2위 | 134초 | 0.2454 | 실패 |
| 3위 | 20초 | 0.2446 | 성공 |
| 4위 | 62초 | 0.2438 | 실패 |
| 5위 | 96초 | 0.2435 | 실패 |

Multilingual CLIP은 실제 정답 구간에 해당하는 약 16초 프레임을 1위로 검색하였다.

또한 약 20초 프레임도 3위로 검색하였다.

반면 기존 Moment-DETR의 두부 자르기 Visual Top-1 구간은 약 101~119초로 정답과 겹치지 않았다.

이는 한국어 Multilingual CLIP이 해당 사례에서 유용한 관련 프레임을 찾을 수 있다는 초기 신호다.

다만 Moment-DETR과 Multilingual CLIP은 출력 및 평가 방식이 다르므로, 이 사례만으로 Multilingual CLIP이 Moment-DETR보다 우수하다고 결론 내릴 수는 없다.

또한 1위와 2위의 코사인 유사도 차이는 약 0.0023으로 작았다.

### 5.2 기존 6개 검색어 전체 평가

| 검색 장면 | 정답 구간 | Multilingual CLIP 1위 | Top-1 성공 |
|---|---|---|---|
| 두부 자르기 | 12~23초 | 16초 | 성공 |
| 두부 양념 준비 | 95~135초 | 20초 | 실패 |
| 가위로 소스 짜기 | 5.96~11.32초 | 36초 | 실패 |
| 쌀 씻기 | 29~32초 | 36초 | 실패 |
| 다이소 파우치 | 3.08~8.40초 | 6초 | 성공 |
| 다이소 시트 세제 | 13~18초 | 30초 | 실패 |

전체 성능은 다음과 같았다.

| 평가 지표 | Multilingual CLIP |
|---|---:|
| Top-1 Point Hit | 33.3% (2/6) |
| Top-5 Point Hit | 83.3% (5/6) |

### 5.3 결과 해석

Multilingual CLIP은 두부 자르기와 다이소 파우치 사례에서 정답 시간대의 프레임을 1위로 검색하였다.

또한 6개 중 5개 검색어에서는 상위 5개 안에 정답 시간대의 프레임이 존재하였다.

그러나 가위로 소스 짜기에서는 상위 5개 프레임 모두 정답 시간대와 일치하지 않았다.

**결론적으로 Multilingual CLIP은 일부 검색어에서 관련 프레임 검색에 성공했지만, 전체 검색어에서 안정적인 Top-1 성능은 확인되지 않았다.**

---

## 6. 기존 영어 CLIP과 동일 조건 비교

### 6.1 비교 목적

Multilingual CLIP이 효과적인지 확인하기 위해 기존 영어 CLIP과 동일한 프레임 검색 방식으로 비교하였다.

기존 Moment-DETR 전체를 사용하는 대신 영어 CLIP의 `pooler_output`을 활용하였다.

두 모델 모두 동일한 기존 영상 CLIP 특징에 대해 코사인 유사도를 계산하였다.

### 6.2 비교 조건

| 항목 | 영어 CLIP | Multilingual CLIP |
|---|---|---|
| 검색어 언어 | 영어 | 한국어 |
| 텍스트 임베딩 | CLIP pooler_output | Multilingual CLIP 문장 임베딩 |
| 출력 차원 | 512 | 512 |
| 영상 특징 | 동일 | 동일 |
| 프레임 간격 | 약 2초 | 약 2초 |
| 검색 방법 | 코사인 유사도 | 코사인 유사도 |
| 평가 지표 | Top-1·Top-5 Point Hit | 동일 |

이 실험을 통해 두 검색 경로를 동일한 프레임 검색 기준에서 비교하였다.

### 6.3 검색어별 Top-1 결과

| 검색 장면 | 영어 CLIP 1위 | 영어 성공 | Multilingual 1위 | 다국어 성공 |
|---|---:|---|---:|---|
| 두부 자르기 | 52초 | 실패 | 16초 | 성공 |
| 두부 양념 준비 | 52초 | 실패 | 20초 | 실패 |
| 가위로 소스 짜기 | 8초 | 성공 | 36초 | 실패 |
| 쌀 씻기 | 36초 | 실패 | 36초 | 실패 |
| 다이소 파우치 | 6초 | 성공 | 6초 | 성공 |
| 다이소 시트 세제 | 14초 | 성공 | 30초 | 실패 |

### 6.4 전체 성능 비교

| 평가 지표 | 영어 CLIP | Multilingual CLIP |
|---|---:|---:|
| Top-1 Point Hit | **50.0% (3/6)** | 33.3% (2/6) |
| Top-5 Point Hit | 83.3% (5/6) | 83.3% (5/6) |

영어 CLIP이 Multilingual CLIP보다 Top-1 Point Hit에서 16.7%p 높았다.

Top-5 Point Hit은 두 모델이 동일하였다.

따라서 **이번 6개 개발 사례에서는 Multilingual CLIP이 기존 영어 CLIP보다 전반적으로 우수하다는 근거가 확인되지 않았다.**

단, 평가 표본이 작으므로 이 차이를 일반적인 성능 격차로 해석해서는 안 된다.

---

## 7. Multilingual CLIP의 상호보완 가능성

### 7.1 검색어별 성공 방식의 차이

두 모델을 비교한 결과, 일부 검색어에서 성공 여부가 달랐다.

| 구분 | 해당 검색어 | 개수 |
|---|---|---:|
| 두 모델 모두 Top-1 성공 | 다이소 파우치 | 1 |
| 영어 CLIP만 Top-1 성공 | 가위로 소스 짜기, 다이소 시트 세제 | 2 |
| Multilingual CLIP만 Top-1 성공 | 두부 자르기 | 1 |
| 두 모델 모두 Top-1 실패 | 두부 양념 준비, 쌀 씻기 | 2 |

이 결과는 영어 CLIP과 Multilingual CLIP이 항상 동일한 장면을 우선 선택하는 것은 아니라는 점을 보여준다.

특히 두부 자르기는 영어 CLIP이 실패했지만 Multilingual CLIP이 성공하였다.

반면 가위로 소스 짜기와 다이소 시트 세제는 영어 CLIP이 성공했지만 Multilingual CLIP이 실패하였다.

따라서 어느 한 검색 경로를 모든 검색어에서 무조건 우선하는 방식은 적절하지 않을 수 있다.

### 7.2 두 검색 경로의 후보를 함께 활용한다면?

기존 39단계 결과를 사후적으로 비교하면 다음과 같다.

| 진단 기준 | 성공 사례 |
|---|---:|
| 두 모델 중 하나라도 Top-1 프레임 성공 | 4/6 (66.7%) |
| 두 모델 중 하나라도 Top-5에 정답 프레임 존재 | 6/6 (100.0%) |

**이 수치는 실제 결합 모델의 성능이 아니다.**

정답을 알고 있는 상태에서 두 모델 중 하나라도 정답을 찾았는지를 확인한 사후 진단용 수치이다.

실제 서비스에서는 어느 모델의 후보를 선택할지 정답 없이 판단해야 하므로, 두 결과를 합친다고 자동으로 66.7% 또는 100%의 정확도를 얻는 것은 아니다.

다만 Multilingual CLIP을 기존 영어 CLIP의 완전한 대체 모델로 사용하기보다는, **서로 다른 후보를 생성하는 보조 검색 경로로 활용할 가능성**은 확인하였다.

---

## 8. 현재 Multilingual CLIP 실험의 한계 및 문제점

### 8.1 문제 ① Top-1 검색 성능의 불안정성

Multilingual CLIP은 일부 검색어에서는 정확한 장면을 찾았지만 다른 검색어에서는 무관한 시간대의 프레임을 1위로 반환하였다.

특히 다음 사례에서 실패하였다.

- 가위로 소스 짜기: 정답 5.96~11.32초, 예측 36초
- 쌀 씻기: 정답 29~32초, 예측 36초
- 다이소 시트 세제: 정답 13~18초, 예측 30초

따라서 기존 영어 CLIP을 즉시 교체할 근거가 부족하다.

### 8.2 문제 ② 프레임 검색과 구간 검색의 차이

이번 실험은 프레임을 검색했을 뿐, 관련 장면의 시작·종료 구간을 생성하지 않았다.

예를 들어 16초 프레임이 정답 장면 안에 있다고 하더라도, 실제로 12~23초 구간을 찾아낸 것은 아니다.

PinPoint AI 서비스에서는 최종적으로 다음 형태의 결과가 필요하다.

```json
{
  "start_time": 12.0,
  "end_time": 23.0
}
```

따라서 Multilingual CLIP을 실제 검색 서비스에 활용하려면 프레임 점수를 시간 구간으로 변환하는 과정이 필요하다.

### 8.3 문제 ③ 기존 Moment-DETR과의 입력 구조 차이

기존 Moment-DETR은 CLIP 토큰별 특징을 사용한다.

실제 영어 검색어의 입력 형태는 `(7, 512)`로 확인되었다.

반면 이번 Multilingual CLIP 실험은 문장 전체 임베딩 `(1, 512)`를 사용한다.

두 출력 모두 512차원 특징을 포함하지만, 토큰 수와 표현 방식이 다르다.

따라서 Multilingual CLIP 문장 임베딩을 기존 Moment-DETR의 입력에 그대로 넣는 것은 학습 당시 입력 조건과 달라질 수 있다.

현재까지는 직접 교체 방식의 호환성과 성능이 검증되지 않았다.

### 8.4 문제 ④ 기존 영상 특징과의 정렬 수준 미검증

Multilingual CLIP은 CLIP 계열 임베딩 공간에 문장을 연결하도록 학습된 모델이지만, 이번에 사용한 PinPoint 영상 특징과의 실제 정렬 수준은 충분히 검증되지 않았다.

특히 영상 특징의 추출 방식, 전처리, 정규화, 학습 시 사용한 CLIP 구현에 따라 검색 결과에 차이가 발생할 수 있다.

따라서 출력 차원이 동일하다는 이유만으로 완전한 호환성을 주장해서는 안 된다.

### 8.5 문제 ⑤ 관련도 점수의 해석 한계

두부 자르기 사례에서 Multilingual CLIP의 점수는 다음과 같았다.

- 1위: 0.2477
- 2위: 0.2454
- 3위: 0.2446

상위 후보 간 점수 차이가 작다.

코사인 유사도는 모델이 출력한 관련도 지표이며, 정답일 확률을 의미하지 않는다.

따라서 유사도 0.2477을 검색 정확도 24.77%로 해석할 수 없다.

또한 서로 다른 CLIP 모델의 코사인 유사도 원점수를 그대로 비교하여 최종 후보를 선택하는 방식은 충분한 검증이 필요하다.

### 8.6 문제 ⑥ 평가 데이터의 제한

이번 실험은 3개 영상의 6개 검색어만 사용하였다.

모두 이전 실험에서 사용했던 개발 사례이므로 독립적인 테스트 데이터가 아니다.

따라서 현재 결과는 모델 선택과 개선 방향을 검토하기 위한 초기 실험 결과로 해석한다.

---

## 9. 향후 개선 방향

### 9.1 방향 ① 영어 CLIP을 즉시 교체하지 않기

현재 실험에서는 영어 CLIP의 Top-1 Point Hit이 더 높았다.

따라서 Multilingual CLIP을 기존 Visual 검색 경로의 대체 모델로 즉시 적용하지 않는다.

기존 Moment-DETR과 영어 CLIP은 기준선으로 유지한다.

### 9.2 방향 ② Multilingual CLIP의 독립적인 검색 경로 검토

다음과 같은 병렬 검색 구조를 검토한다.

```text
사용자 자연어 검색어
        │
        ├── 영어 CLIP
        │       ↓
        │   프레임 관련도 검색
        │
        └── Multilingual CLIP
                ↓
            한국어 프레임 관련도 검색
                │
                ↓
        후보 프레임 결합 및 비교
                ↓
        시간 구간 후보 생성
                ↓
        최종 재순위화
```

이 구조는 아직 구현되지 않은 후속 실험 계획이다.

서로 다른 검색 경로에서 나온 점수와 후보를 어떤 방식으로 결합할지는 추가 검증이 필요하다.

### 9.3 방향 ③ 프레임 검색 결과의 구간화

현재는 관련도 점수가 높은 개별 프레임만 확보하고 있다.

다음 실험에서는 이를 실제 영상 구간으로 변환하는 방법을 검토한다.

검토할 방법은 다음과 같다.

- 인접한 프레임의 관련도 점수를 활용한 구간 생성
- 일정 시간 범위의 유사도 평균 또는 집계 점수 활용
- 높은 관련도를 보이는 연속 프레임을 하나의 구간으로 병합
- 기존 Moment-DETR 구간과 CLIP 프레임 관련도 결합
- 후보 구간의 시작·종료 경계 보정

단, 특정 프레임을 중심으로 임의의 길이만큼 구간을 확장하는 방식은 성능 향상을 보장하지 않는다.

프레임을 구간으로 변환한 후에는 기존 평가와 동일하게 Temporal IoU를 계산하여 성능을 비교한다.

### 9.4 방향 ④ 동일한 구간 검색 지표로 최종 비교

향후에는 다음 방식들을 동일한 조건에서 비교한다.

1. 기존 Moment-DETR Visual
2. 영어 CLIP 프레임 기반 구간 검색
3. Multilingual CLIP 프레임 기반 구간 검색
4. 필요한 경우 두 CLIP 검색 경로의 후보 결합
5. 기존 Moment-DETR·MiniLM·Consensus와의 결합

평가 지표는 다음을 사용한다.

- Top-1 IoU
- R1@0.5
- R5@0.5
- 후보 전체 Oracle@0.5
- 필요할 경우 시작 시간 오차

단, 프레임 검색 결과를 구간으로 변환하는 방식 자체도 검증해야 하며, 단순 Point Hit을 R1@0.5와 혼동하지 않는다.

### 9.5 방향 ⑤ Consensus V4와의 연계

Multilingual CLIP으로 새로운 후보를 확보하더라도 최종 Top-1 결정 문제는 남아 있다.

따라서 필요하다면 다음 검색 경로를 종합하는 Consensus V4를 검토한다.

- 기존 Visual Moment-DETR
- Fine-tuned MiniLM-KO
- Fine-tuned MiniLM-EN
- Multilingual CLIP 기반 Visual 후보

단, 경로를 무조건 많이 추가하면 중복 후보가 늘어나거나 잘못된 후보의 영향력이 커질 수 있다.

따라서 Multilingual CLIP의 추가가 실제 성능 개선에 기여하는지 먼저 검증하고, 유효한 경우에만 Consensus 구조에 반영한다.

---

## 10. 다음 실험 계획

| 순서 | 실험 내용 | 목적 | 상태 |
|---|---|---|---|
| 1 | 기존 Moment-DETR과 CLIP 입력 구조 확인 | 다국어 모델 연결 조건 점검 | 완료 |
| 2 | Multilingual CLIP 로딩 및 두부 검색 | 한국어 프레임 검색 가능성 확인 | 완료 |
| 3 | 6개 검색어 Multilingual CLIP 평가 | 기본 검색 성능 파악 | 완료 |
| 4 | 영어 CLIP과 동일 조건 비교 | 상대적인 프레임 검색 성능 비교 | 완료 |
| 5 | 영어·다국어 CLIP 후보 상호보완성 분석 | 보조 검색 경로 활용 가능성 검토 | 일부 사후 지표 확인, 상세 분석 예정 |
| 6 | 프레임 검색 결과의 구간화 방식 검토 | start_time / end_time 생성 | 예정 |
| 7 | 구간 IoU 기반 성능 평가 | 기존 Moment-DETR과 비교 가능한 성능 측정 | 예정 |
| 8 | 검색 경로 결합 실험 | 후보 생성 및 최종 Top-1 개선 여부 확인 | 예정 |
| 9 | Consensus V4 검토 | 후보 재순위화 개선 | 보류 |
| 10 | 추가 검색어·영상 검증 | 일반화 성능 검증 | 예정 |

### 다음 단계에서 우선 확인할 사항

현재까지는 두 모델이 서로 다른 검색어에서 성공한다는 사실을 확인하였다.

다음에는 **두 모델의 프레임 후보를 실제 시간 구간으로 만드는 방법이 타당한지** 검토하는 것이 중요하다.

특히 기존 Moment-DETR의 후보 시간 구간과 CLIP 프레임 관련도를 함께 활용할 수 있는지도 확인한다.

이 과정에서도 정답 구간은 추론 과정에 사용하지 않고 평가에만 사용한다.

---

## 11. 실험 결과 및 파일 관리

이번 Multilingual CLIP 실험의 모든 결과는 다음 경로를 기준으로 관리한다.

`/content/drive/MyDrive/pinpoint_capstone/04_PinPoint_ConsensusV3_MultilingualCLIP/`

### 11.1 주요 결과 파일

**36단계 — 두부 영상 Multilingual CLIP 첫 검색**

`02_results/tofu_multilingual_clip_frame_search_step36.json`

**37단계 — Multilingual CLIP 6개 검색어 평가**

`03_evaluation/multilingual_clip_6cases_frame_evaluation_step37.json`

**39단계 — 영어 CLIP vs Multilingual CLIP 비교**

`03_evaluation/english_vs_multilingual_clip_6cases_step39.json`

**기존 Consensus V1·V2·V3 비교 결과**

`03_evaluation/consensus_v1_v2_v3_6cases_step32.json`

**Consensus V3 실패 원인 분석**

`03_evaluation/consensus_v3_failure_diagnosis_step33.json`

### 11.2 모델 저장

Multilingual CLIP 모델은 다음 실험 폴더의 Hugging Face 캐시 경로를 사용하여 다운로드하였다.

`04_models/huggingface_cache/`

기존 Moment-DETR 및 Fine-tuned MiniLM 체크포인트는 수정하지 않았다.

### 11.3 관리 원칙

- 기존 모델 체크포인트 보존
- 기존 Fusion 및 Consensus 코드 유지
- 기존 결과 JSON 덮어쓰기 금지
- 새 실험 결과는 별도 파일로 저장
- GitHub `main` 브랜치 직접 수정 금지
- 성공과 실패 사례 모두 기록
- 개발 사례와 독립적인 평가 사례 구분
- 정답 정보를 이용한 사후 분석과 실제 추론 성능을 구분

---

## 12. 최종 결론

이번 실험에서는 한국어 Multilingual CLIP을 이용해 기존 영상 CLIP 특징과 한국어 검색어의 유사도를 계산하고, 관련 프레임을 검색하는 데 성공하였다.

특히 두부 자르기 검색에서 정답 구간에 포함되는 16초 프레임을 1위로 찾아, 한국어 기반 Visual 검색의 가능성을 확인하였다.

그러나 기존 6개 검색어 전체를 평가한 결과, Multilingual CLIP의 Top-1 Point Hit은 33.3%로 기존 영어 CLIP의 50.0%보다 낮았다.

Top-5 Point Hit은 두 모델 모두 83.3%로 동일하였다.

또한 두 모델은 서로 다른 검색어에서 성공하는 경향을 보였다. 두 모델 중 하나라도 Top-1 검색에 성공한 사례는 4개, Top-5에서 성공한 사례는 6개였다.

이 수치는 실제 결합 모델의 정확도가 아니라 정답을 이용한 사후 진단 결과이지만, 두 검색 경로가 상호보완적인 후보를 생성할 가능성을 보여준다.

**따라서 현재 단계에서는 Multilingual CLIP이 기존 영어 CLIP이나 Moment-DETR을 대체할 만큼 우수하다는 근거가 부족하다.**

향후에는 기존 모델을 유지한 상태에서 Multilingual CLIP의 프레임 후보를 시간 구간으로 변환하고, 기존 Visual 검색 및 자막 검색 경로와 결합했을 때 실제 Top-1 구간 검색 성능이 개선되는지 확인할 계획이다.

이후 필요하다면 Consensus V4를 통해 최종 후보의 재순위화 방식을 추가 개선한다.

### 현재 판단 요약

- **Multilingual CLIP 모델 로딩 및 한국어 프레임 검색:** 성공
- **기존 영어 CLIP 대비 전체 Top-1 성능 향상:** 확인되지 않음
- **서로 다른 검색어에서의 상호보완 가능성:** 일부 확인
- **기존 Moment-DETR 직접 교체:** 현재 권장하지 않음
- **독립적인 한국어 Visual 검색 경로 활용:** 추가 실험 가치 있음
- **시간 구간 검색 정확도 개선:** 아직 검증되지 않음
- **향후 우선 과제:** 프레임 후보의 구간화 및 동일 지표 기반 비교
- **Consensus V4:** 후속 개선 계획으로 유지

**최종 목표는 Multilingual CLIP을 추가하는 것 자체가 아니라, 사용자 검색어에 가장 적합한 영상 구간을 정확하게 Top-1으로 반환하도록 PinPoint AI의 검색 성능을 개선하는 것이다.**